# Mission 3 ver14 — 오심 오류 분석 → 학습 가중치 → attention·대조학습 비교

**이 노트북 한 개와 기존 `mission3_ver5_inputs.zip`으로 실행합니다.**
Colab의 GPU 런타임을 선택하고 **런타임 → 모두 실행**을 누르세요.
Drive 연결 요청에는 ZIP 바로가기가 있는 **공용 계정**을 선택합니다.

Training 전사와 정답을 먼저 점검한 뒤, 같은 Training 내부 분할에서 6개 후보를 비교합니다.
최종 선택은 Training 지표로만 결정하고, 선택된 구성 하나를 전체 Training으로 새로 학습합니다.
공식 Validation은 마지막에 한 번 평가합니다. 예측은 증상별 두 class의 `argmax`입니다.

**오심 F1과 전체 Macro F1의 개선은 실제 실행 결과로 확인해야 합니다.**
사후 threshold 조정이나 원본 정답 수정은 하지 않습니다.
GPU 학습 6회와 최종 재학습 1회이므로 수 시간이 필요합니다.
비슷한 GPU에서 약 6–10시간을 예상하지만 GPU·조기 종료에 따라 달라집니다. 고정 시간 제한은 없습니다.


## 원본 검토에서 확인한 문제

기존 오류 검토 표 80행, 중복을 제외한 76개 통화를 원본과 대조했습니다.
아래는 개인정보와 통화 식별자를 제외한 증상 내용의 요약입니다.

| 원본에서 관찰한 증상 내용 | 제공된 정답과 모델 결과 | 분석 방향 |
| --- | --- | --- |
| 메스꺼움과 구토를 함께 말한 통화 | 정답은 구토·어지러움, 모델의 오심은 FP | 표현이 명시되어도 정답과 어긋나는 사례를 검토 |
| 토할 것 같은 느낌과 어지러움을 말한 통화 | 정답은 어지러움, 모델의 오심은 FP | 구토·오심·어지러움의 표현과 정답 경계를 비교 |
| 토할 것 같은 느낌을 말한 통화 | 정답에 오심이 있지만 모델은 구토만 예측하여 FN | 유사 표현에서 구토와 오심을 구별하는 학습을 점검 |

같은 표현이 서로 다른 정답 조합에 나타납니다. 이 관찰만으로 라벨 오류를 확정할 수 없으며,
**원본 정답은 그대로 유지합니다.** 질문·부정 표현도 단순 존재 여부로 증상을 단정하지 않습니다.

기존 ver13의 Training holdout 오심 FN 575개 중 567개(98.6%)가 단일 512-token window였습니다.
ver11에서 맞혔다가 ver13이 놓친 16개도 모두 단일 window였고 최대 417 tokens였습니다.
정규식으로 찾은 오심 표현 후보는 모두 첫 window 안에 있었습니다.
따라서 긴 대화 확장보다 정답의 경계·학습 손실·증상별 정밀도와 재현율을 먼저 점검합니다.
정규식에 포함되지 않은 우회 표현은 별도 사람이 검토해야 합니다.


## 1 → 2 → 3 실행 순서

1. **Training 원본 점검:** 중복·정답 분포·오심/구토 조합·질문/부정 후보를 집계하고 검토 자료를 저장합니다.
   fit/holdout을 구분하며 검토 목록에는 원본 행 위치·정답·후보 표시를 저장합니다. 화면에는 집계만 표시합니다.
2. **오심 양성 학습 가중치 비교:** attention 없이 교차엔트로피를 사용하고 가중치 1·2·3을 비교합니다.
   1이 이번 실행의 새 기준 모델입니다. 이는 학습 손실의 비용 변경이며 사후 예측 보정이 아닙니다.
3. **구조와 손실을 분리해 비교:** 2단계에서 고른 가중치를 고정하고 아래 3개를 같은 조건으로 새로 학습합니다.

| 후보 | Attention | 교차엔트로피 | 대조학습 |
| --- | --- | --- | --- |
| attention_off_ce | 끔 | 사용 | 없음 |
| attention_on_ce | 켬 | 사용 | 없음 |
| attention_on_supcon | 켬 | 사용 | 0.1 |

3단계 세 후보는 실제 batch 8, accumulation 2로 맞춥니다.
세 후보 모두 동일 통화를 dropout으로 두 번 읽고 두 교차엔트로피의 평균을 학습합니다.
attention_on_supcon에만 대조학습을 더해 두 view 사용 자체와 대조학습 효과를 분리합니다.
2단계는 batch 4, accumulation 4이며 두 단계 모두 유효 batch는 16입니다.
대조학습의 비교 표본은 실제 batch 안의 통화이므로 accumulation으로 늘어나지 않습니다.
encoder·Training 분할·seed·학습률·epoch 선택 규칙도 동일하게 유지합니다.

최종 후보는 새 기준 모델의 **Macro F1과 오심 F1을 모두 유지하거나 개선한 후보**에서 Macro F1 우선으로 고릅니다.
조건을 만족하는 개선 후보가 없으면 이번에 학습한 기준 모델을 유지합니다.
이후 선택된 구성만 공개 초기 가중치에서 전체 Training으로 한 번 다시 학습하고 Validation을 평가합니다.


In [ ]:
# @title 실행 설정
SETTINGS = {'version': 'ver14', 'backend': 'neural', 'model_id': 'klue/roberta-base', 'revision': '02f94ba5e3fcb7e2a58a390b8639b0fac974a8da', 'label_attention': False, 'contrastive_weight': 0.0, 'input_zip': '/content/drive/MyDrive/mission3_ver5_inputs.zip', 'output_dir': '/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver14', 'work_dir': '/content/mission3_ver14', 'train_dir': '', 'validation_dir': '', 'run_validation': True, 'seed': 42, 'holdout_size': 0.2, 'max_length': 512, 'overlap': 64, 'batch_size': 4, 'encoder_window_batch': 4, 'gradient_accumulation': 4, 'max_epochs': 4, 'patience': 2, 'learning_rate': 2e-05, 'head_learning_rate': 0.0001, 'refit_full_training': True, 'allow_cpu': False, 'log_every': 250, 'nausea_weight_candidates': [1.0, 2.0, 3.0], 'stage3_batch_size': 8, 'stage3_gradient_accumulation': 2, 'resume_run': ''}
# Keep both explicit label directories empty when using the existing input ZIP.
# To resume, set resume_run to the complete previously created run directory.
# Resume checks source, data, and settings fingerprints before reusing completed candidates.
# GPU memory: adjust each stage's microbatch and accumulation together.
# SupCon compares only examples in the actual microbatch.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
# @title 실행 코드 준비 (다른 파일 업로드 불필요)
import tempfile
from pathlib import Path
RUNTIME_SOURCES = {'__init__.py': '"""Standalone Mission 3 categorical experiments embedded in ver10-ver13."""\n', 'data.py': '"""Text-only Mission 3 data, grouped Training holdout, and discrete predictions."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport re\nimport tempfile\nimport unicodedata\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\nLEXICAL_PATTERNS = {\n    "nausea": r"오심|메스꺼|메스껍|메슥|매스꺼|매스껍|매슥|울렁|구역|미식|느글|토할",\n    "question": r"\\?|나요|까요|습니까|신가|세요",\n    "negation": r"아니|없|않|못|안\\s|아닌",\n}\nLEXICAL_COUNTS_NOTE = (\n    "Regex matches are review candidates only: questions and negation may appear elsewhere "\n    "in the dialogue. These flags do not establish annotation errors or modify labels/predictions."\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """Read only utterance text, keeping turn boundaries and chronological order."""\n    key = "utterances" if "utterances" in annotation else "utterences"\n    utterances = annotation.get(key)\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if not isinstance(utterance, dict) or not isinstance(utterance.get("text"), str):\n            raise ValueError("Each utterance must be an object with string text")\n        text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n        if text:\n            texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(item, str) for item in raw):\n        raise ValueError("symptom must be a string, a list of strings, or null")\n    found = {item.strip() for item in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir, include_targets: bool = False):\n    label_dir = Path(label_dir)\n    if not label_dir.is_dir():\n        raise ValueError(f"Label directory does not exist: {label_dir}")\n    paths = sorted(path for path in label_dir.rglob("*.json") if path.is_file())\n    if not paths:\n        raise ValueError(f"No label JSON files: {label_dir}")\n    if len({path.name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate JSON filenames make submission identifiers ambiguous")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"Expected a JSON object: {path}")\n        # Inference never inspects symptom, even when a supplied value is malformed.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=8) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef _binary(values, rows=None, name="predictions"):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and values.shape[0] != rows)\n            or not np.isin(values, [0, 1]).all()):\n        raise ValueError(f"{name} must have shape (samples, 9) and contain only discrete 0/1 values")\n    return values.astype(np.int8)\n\n\ndef _greedy_groups(features, sizes, holdout_size, seed):\n    """Dependency-free iterative stratification of indivisible transcript groups.\n\n    This balances label presence and absence, then row counts. Group constraints\n    mean neither this fallback nor iterstrat can guarantee exact proportions.\n    """\n    rng = np.random.default_rng(seed)\n    active = np.ones(len(features), dtype=bool)\n    assignment = np.full(len(features), -1, dtype=np.int8)\n    remaining = features.sum(axis=0).astype(float)\n    proportions = np.asarray([1.0 - holdout_size, holdout_size])\n    desired_labels = proportions[:, None] * remaining[None, :]\n    desired_rows = proportions * sizes.sum()\n    coverage = np.zeros((2, features.shape[1]), dtype=bool)\n    while active.any():\n        available = np.flatnonzero(remaining > 0)\n        if not len(available):\n            candidates = np.flatnonzero(active)\n            rng.shuffle(candidates)\n            label = None\n        else:\n            rare = available[remaining[available] == remaining[available].min()]\n            label = int(rng.choice(rare))\n            candidates = np.flatnonzero(active & features[:, label].astype(bool))\n            rng.shuffle(candidates)\n        for group in candidates:\n            deficits = desired_rows if label is None else desired_labels[:, label]\n            sides = np.flatnonzero(deficits == deficits.max())\n            if len(sides) > 1:\n                sides = sides[desired_rows[sides] == desired_rows[sides].max()]\n            present = np.flatnonzero(features[group])\n            final_occurrences = present[remaining[present] == 1]\n            required = np.unique(np.where(~coverage[:, final_occurrences])[0])\n            if len(required) == 1:\n                # Reserve the last group for a side still missing this target.\n                sides = required\n            side = int(rng.choice(sides))\n            assignment[group] = side\n            desired_rows[side] -= sizes[group]\n            desired_labels[side] -= features[group]\n            remaining -= features[group]\n            coverage[side] |= features[group].astype(bool)\n            active[group] = False\n    return np.flatnonzero(assignment == 0), np.flatnonzero(assignment == 1)\n\n\ndef make_split(texts, targets, holdout_size: float = .2, seed: int = 42):\n    """Split Training only; normalized duplicate transcripts cannot cross sides."""\n    if not 0 < holdout_size < .5:\n        raise ValueError("holdout_size must be in (0, .5)")\n    targets = _binary(targets, len(texts), "targets")\n    if any(not isinstance(text, str) for text in texts):\n        raise TypeError("texts must contain strings only")\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse, sizes = np.unique(groups, return_inverse=True, return_counts=True)\n    presence = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    absence = np.zeros_like(presence)\n    np.maximum.at(presence, inverse, targets)\n    np.maximum.at(absence, inverse, 1 - targets)\n    features = np.concatenate([presence, absence], axis=1)\n    if len(unique) < 2 or np.any(features.sum(axis=0) < 2):\n        raise ValueError("Each symptom needs positive and negative examples in at least two text groups")\n    try:\n        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit\n    except ImportError:\n        MultilabelStratifiedShuffleSplit = None\n    best = None\n    # Retry only group allocation, never inspecting official Validation.\n    for attempt in range(8):\n        current_seed = seed + attempt\n        if MultilabelStratifiedShuffleSplit is None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n        else:\n            splitter = MultilabelStratifiedShuffleSplit(\n                n_splits=1, test_size=holdout_size, random_state=current_seed)\n            fit_groups, holdout_groups = next(splitter.split(np.arange(len(unique)), features))\n        fit = np.flatnonzero(np.isin(inverse, fit_groups))\n        holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n        valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                    and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if not valid and MultilabelStratifiedShuffleSplit is not None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n            fit = np.flatnonzero(np.isin(inverse, fit_groups))\n            holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n            valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                        and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if valid:\n            ratio = targets[holdout].sum(axis=0) / targets.sum(axis=0)\n            discrepancy = float(np.abs(ratio - holdout_size).mean()\n                                + abs(len(holdout) / len(targets) - holdout_size))\n            if best is None or discrepancy < best[0]:\n                best = (discrepancy, fit, holdout)\n    if best is None:\n        raise ValueError("Cannot split text groups with positive and negative examples for every symptom")\n    return best[1], best[2]\n\n\ndef _atomic_text(path, write, encoding="utf-8-sig"):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding=encoding, newline="",\n                                         dir=path.parent, prefix=".write_", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            write(handle)\n        temporary.replace(path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef write_json(path, value):\n    _atomic_text(path, lambda handle: json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False),\n                 encoding="utf-8")\n\n\ndef write_predictions(paths, predictions, output):\n    """Write already discrete model decisions; this API has no score or threshold."""\n    predictions = _binary(predictions, len(paths))\n    if len({Path(path).name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate prediction identifiers")\n\n    def write(handle):\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predictions):\n            writer.writerow([Path(path).name, str([name for name, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    _atomic_text(output, write)\n    return predictions\n\n\ndef score_predictions(targets, predictions):\n    targets = _binary(targets, name="targets")\n    predictions = _binary(predictions, len(targets))\n    if not len(targets):\n        raise ValueError("Cannot score an empty dataset")\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "micro_f1": float(f1_score(targets, predictions, average="micro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i]),\n                                "false_positive": int(((targets[:, i] == 0) & (predictions[:, i] == 1)).sum()),\n                                "false_negative": int(((targets[:, i] == 1) & (predictions[:, i] == 0)).sum())}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n\n\ndef training_diagnostics(texts, targets):\n    """Return aggregate counts only, including suspected nausea annotation ambiguity."""\n    targets = _binary(targets, len(texts), "targets")\n    groups = {}\n    for index, text in enumerate(texts):\n        groups.setdefault(text_group(text), []).append(index)\n    duplicates = [indices for indices in groups.values() if len(indices) > 1]\n    conflicts = [indices for indices in duplicates if len(np.unique(targets[indices], axis=0)) > 1]\n    nausea = targets[:, TARGET_SYMPTOMS.index("오심")].astype(bool)\n    vomiting = targets[:, TARGET_SYMPTOMS.index("구토")].astype(bool)\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    lexical = {key: int(flags.sum()) for key, flags in signals.items()}\n    for key in ("question", "negation"):\n        cooccurrence = signals["nausea"] & signals[key]\n        lexical[f"nausea_and_{key}"] = int(cooccurrence.sum())\n        lexical[f"nausea_and_{key}_label_positive"] = int((cooccurrence & nausea).sum())\n        lexical[f"nausea_and_{key}_label_negative"] = int((cooccurrence & ~nausea).sum())\n    lexical["nausea_mention_label_positive"] = int((signals["nausea"] & nausea).sum())\n    lexical["nausea_mention_label_negative"] = int((signals["nausea"] & ~nausea).sum())\n    return {\n        "samples": len(texts), "all_zero_samples": int((targets.sum(axis=1) == 0).sum()),\n        "empty_text_samples": sum(not text.strip() for text in texts),\n        "positive_counts": {name: int(targets[:, i].sum()) for i, name in enumerate(TARGET_SYMPTOMS)},\n        "text_groups": len(groups), "duplicate_groups": len(duplicates),\n        "duplicate_extra_samples": sum(len(indices) - 1 for indices in duplicates),\n        "conflicting_label_groups": len(conflicts),\n        "conflicting_label_samples": sum(len(indices) for indices in conflicts),\n        "nausea_vomiting_contingency": {\n            "neither": int((~nausea & ~vomiting).sum()), "nausea_only": int((nausea & ~vomiting).sum()),\n            "vomiting_only": int((~nausea & vomiting).sum()), "both": int((nausea & vomiting).sum())},\n        "lexical_cooccurrence_counts": lexical,\n        "lexical_patterns": LEXICAL_PATTERNS,\n        "lexical_counts_note": LEXICAL_COUNTS_NOTE,\n    }\n\n\ndef export_errors(paths, texts, targets, predictions, output_dir, max_per_label: int = 20):\n    """Save full text locally for manual review; never print sensitive dialogue."""\n    targets = _binary(targets, len(texts), "targets")\n    predictions = _binary(predictions, len(texts))\n    if len(paths) != len(texts) or type(max_per_label) is not int or max_per_label < 1:\n        raise ValueError("paths/texts must align and max_per_label must be a positive integer")\n    output_dir = Path(output_dir)\n    outputs = []\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    header = ["label file name", "symptom", "error_type", "true_symptoms", "predicted_symptoms", "text",\n              "review_category", "review_note", "has_nausea_expression", "has_question_expression",\n              "has_negation_expression"]\n\n    def review_row(index, symptom, category):\n        actual = [name for name, flag in zip(TARGET_SYMPTOMS, targets[index]) if flag]\n        predicted = [name for name, flag in zip(TARGET_SYMPTOMS, predictions[index]) if flag]\n        return [Path(paths[index]).name, symptom, category, str(actual), str(predicted), texts[index], "", "",\n                int(signals["nausea"][index]), int(signals["question"][index]), int(signals["negation"][index])]\n\n    for label, name in enumerate(TARGET_SYMPTOMS):\n        fn = np.flatnonzero((targets[:, label] == 1) & (predictions[:, label] == 0)).tolist()\n        fp = np.flatnonzero((targets[:, label] == 0) & (predictions[:, label] == 1)).tolist()\n        indices = []\n        while (fn or fp) and len(indices) < max_per_label:\n            for pending in (fn, fp):\n                if pending and len(indices) < max_per_label:\n                    indices.append(pending.pop(0))\n        output = output_dir / f"errors_{name}.csv"\n\n        def write(handle):\n            writer = csv.writer(handle)\n            writer.writerow(header)\n            for index in indices:\n                writer.writerow(review_row(index, name, "FN" if targets[index, label] else "FP"))\n        _atomic_text(output, write)\n        outputs.append(output)\n\n    nausea_index, vomiting_index = TARGET_SYMPTOMS.index("오심"), TARGET_SYMPTOMS.index("구토")\n    nausea, vomiting = targets[:, nausea_index].astype(bool), targets[:, vomiting_index].astype(bool)\n    combinations = {"neither": ~nausea & ~vomiting, "nausea_only": nausea & ~vomiting,\n                    "vomiting_only": ~nausea & vomiting, "both": nausea & vomiting}\n    group_counts = {}\n    for name, mask in combinations.items():\n        row = {"n": int(mask.sum())}\n        for symptom, column in (("nausea", nausea_index), ("vomiting", vomiting_index)):\n            actual, predicted = targets[:, column].astype(bool), predictions[:, column].astype(bool)\n            row[f"predicted_{symptom}"] = int((mask & predicted).sum())\n            row[f"{symptom}_tp"] = int((mask & actual & predicted).sum())\n            row[f"{symptom}_fp"] = int((mask & ~actual & predicted).sum())\n            row[f"{symptom}_fn"] = int((mask & actual & ~predicted).sum())\n            row[f"{symptom}_tn"] = int((mask & ~actual & ~predicted).sum())\n        group_counts[name] = row\n    summary_path = output_dir / "nausea_vomiting_error_groups.json"\n    write_json(summary_path, group_counts)\n    outputs.append(summary_path)\n\n    predicted_nausea = predictions[:, nausea_index].astype(bool)\n    controls = {"TP": nausea & predicted_nausea, "FP": ~nausea & predicted_nausea,\n                "FN": nausea & ~predicted_nausea, "TN": ~nausea & ~predicted_nausea}\n    rng = np.random.default_rng(42)\n    sampled = []\n    for category, mask in controls.items():\n        candidates = np.flatnonzero(mask)\n        chosen = rng.choice(candidates, size=min(10, len(candidates)), replace=False)\n        sampled.extend((int(index), category) for index in chosen)\n    # Mix categories for human review while preserving reproducibility.\n    rng.shuffle(sampled)\n    review_path = output_dir / "review_오심.csv"\n\n    def write_controls(handle):\n        writer = csv.writer(handle)\n        writer.writerow(header)\n        for index, category in sampled:\n            writer.writerow(review_row(index, "오심", category))\n    _atomic_text(review_path, write_controls)\n    outputs.append(review_path)\n    return outputs\n', 'bootstrap.py': '"""Safely restore label JSONs only; old model weights are never extracted."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\nMARKER_NAME = ".mission3_label_bundle_complete.json"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not isinstance(name, str) or not name or "\\\\" in name or ":" in name\n            or path.is_absolute() or ".." in path.parts\n            or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    target = root / _relative(name)\n    if not target.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    # Existing links must not redirect even to another location inside the root.\n    for component in (target, *target.parents):\n        if component == root.parent:\n            break\n        if component.is_symlink():\n            raise ValueError(f"Extraction destination contains a symlink: {name}")\n    return target\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as reader:\n        for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _selected_files(archive):\n    infos = archive.infolist()\n    names = set()\n    entry_types = {}\n    for info in infos:\n        # Windows ZipInfo normalizes backslashes in filename; orig_filename\n        # retains the stored member spelling that must also pass validation.\n        _relative(info.orig_filename)\n        _relative(info.filename)\n        normalized = info.filename.rstrip("/").casefold()\n        if normalized in names:\n            raise ValueError(f"Duplicate ZIP member or directory: {info.filename}")\n        names.add(normalized)\n        entry_types[normalized] = info.is_dir()\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n    for name in entry_types:\n        if any(parent.as_posix() in entry_types and not entry_types[parent.as_posix()]\n               for parent in PurePosixPath(name).parents if parent.as_posix() != "."):\n            raise ValueError("ZIP file/directory collision")\n    selected = [info for info in infos if not info.is_dir() and info.filename.endswith(".json")\n                and info.filename.startswith((TRAIN_PREFIX, VALID_PREFIX))]\n    for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n        subset = [info for info in selected if info.filename.startswith(prefix)]\n        if not subset:\n            raise ValueError(f"Input ZIP has no label JSONs in {prefix}")\n        if len({PurePosixPath(info.filename).name.casefold() for info in subset}) != len(subset):\n            raise ValueError("Duplicate label JSON filenames in input ZIP")\n    by_name = {info.filename: info for info in selected}\n    if "bundle_manifest.json" in archive.namelist():\n        raw = archive.read("bundle_manifest.json")\n        manifest = json.loads(raw)\n        if not isinstance(manifest, dict) or manifest.get("format") != "mission3_ver5_inputs_v1":\n            raise ValueError("Unsupported Mission 3 input bundle manifest")\n        files = manifest.get("files")\n        if not isinstance(files, list):\n            raise ValueError("Bundle manifest has no files")\n        metadata, listed = {}, set()\n        for row in files:\n            if not isinstance(row, dict) or not isinstance(row.get("path"), str):\n                raise ValueError("Invalid bundle manifest file entry")\n            name = row["path"]\n            _relative(name)\n            if name.casefold() in listed:\n                raise ValueError(f"Duplicate manifest entry: {name}")\n            listed.add(name.casefold())\n            if name not in by_name:\n                continue\n            size, digest = row.get("size"), row.get("sha256")\n            if (type(size) is not int or size < 0 or not isinstance(digest, str)\n                    or re.fullmatch(r"[0-9a-f]{64}", digest) is None):\n                raise ValueError(f"Invalid manifest metadata: {name}")\n            if by_name[name].file_size != size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest}\n        if set(metadata) != set(by_name):\n            raise ValueError("Bundle manifest does not list every label JSON")\n        for prefix, key in ((TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")):\n            count = sum(name.startswith(prefix) for name in metadata)\n            if type(manifest.get(key)) is not int or manifest[key] != count:\n                raise ValueError(f"Bundle JSON count mismatch: {key}")\n        signature = hashlib.sha256(raw).hexdigest()\n    else:\n        # Without a manifest, hash only label contents, never the large old models.\n        metadata = {}\n        for name, info in sorted(by_name.items()):\n            digest, size = hashlib.sha256(), 0\n            with archive.open(info) as reader:\n                for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                    digest.update(chunk)\n                    size += len(chunk)\n            if size != info.file_size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest.hexdigest()}\n        signature = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()\n    return [metadata[name] for name in sorted(metadata)], signature\n\n\ndef _validate_directory(directory, name):\n    directory = Path(directory).expanduser().resolve()\n    if not directory.is_dir() or not any(path.is_file() for path in directory.rglob("*.json")):\n        raise ValueError(f"{name} label directory does not contain JSON files: {directory}")\n    return directory\n\n\ndef prepare_data(input_zip, work_dir, train_dir="", validation_dir=""):\n    """Use two explicit label directories, or verify/extract the existing bundle.\n\n    Reuse requires a completion marker plus fresh content hashes. This detects\n    changed files even when their size and modification time are preserved.\n    """\n    if bool(train_dir) != bool(validation_dir):\n        raise ValueError("Set both train_dir and validation_dir, or leave both empty")\n    if train_dir:\n        train = _validate_directory(train_dir, "Training")\n        valid = _validate_directory(validation_dir, "Validation")\n        if train == valid or train.is_relative_to(valid) or valid.is_relative_to(train):\n            raise ValueError("Training and Validation label directories must be distinct and disjoint")\n        return train, valid\n    archive_path = Path(input_zip).expanduser().resolve()\n    if not archive_path.is_file():\n        raise FileNotFoundError(f"Input ZIP is missing: {archive_path}")\n    requested_root = Path(work_dir).expanduser()\n    if requested_root.is_symlink():\n        raise ValueError("Extraction work_dir must not be a symlink")\n    root = requested_root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / MARKER_NAME\n    with zipfile.ZipFile(archive_path) as archive:\n        files, signature = _selected_files(archive)\n        # Resolve every destination before any extraction, including existing links.\n        destinations = {row["path"]: _target(root, row["path"]) for row in files}\n        expected_paths = set(destinations.values())\n        for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n            directory = root / prefix\n            if directory.exists():\n                unexpected = set(directory.rglob("*.json")) - expected_paths\n                if unexpected:\n                    raise ValueError("Extraction directory contains label JSONs from another input bundle")\n        previous = None\n        if marker.exists():\n            if marker.is_symlink():\n                raise ValueError("Completion marker must not be a symlink")\n            try:\n                previous = json.loads(marker.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                previous = None\n        expected = {row["path"]: {"size": row["size"], "sha256": row["sha256"]} for row in files}\n        reuse = (isinstance(previous, dict) and previous.get("signature") == signature\n                 and previous.get("files") == expected)\n        pending = []\n        for row in files:\n            target = destinations[row["path"]]\n            verified = (target.is_file() and target.stat().st_size == row["size"]\n                        and _hash(target) == row["sha256"])\n            if not verified:\n                pending.append(row)\n        if reuse and not pending:\n            return root / TRAIN_PREFIX, root / VALID_PREFIX\n        if marker.exists():\n            marker.unlink()\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Insufficient local disk space to extract label JSONs")\n        for row in pending:\n            target = destinations[row["path"]]\n            target.parent.mkdir(parents=True, exist_ok=True)\n            temporary = None\n            try:\n                digest, size = hashlib.sha256(), 0\n                with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                        dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                    temporary = Path(writer.name)\n                    for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                        writer.write(chunk)\n                        digest.update(chunk)\n                        size += len(chunk)\n                if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                    raise ValueError(f"ZIP content verification failed: {row[\'path\']}")\n                temporary.replace(target)\n            finally:\n                if temporary is not None:\n                    temporary.unlink(missing_ok=True)\n        # Do not publish an incomplete extraction as a reusable data set.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=root,\n                                             prefix=".marker_", suffix=".tmp", delete=False) as writer:\n                temporary = Path(writer.name)\n                json.dump({"signature": signature, "files": expected}, writer, ensure_ascii=False)\n            temporary.replace(marker)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n    return root / TRAIN_PREFIX, root / VALID_PREFIX\n', 'neural.py': '"""Text-only dialogue classifiers with ordinary binary class decisions.\n\nEach symptom has two class logits. Predictions use the winning class directly;\nthere is no probability cutoff search, calibration, or post-hoc label weighting.\nPublic pretrained assets are prepared separately from local checkpoint weights.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom numbers import Real\nfrom pathlib import Path\nimport random\nimport re\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import Dataset\n\n\nNUM_LABELS = 9\nCONTRASTIVE_DIM = 64\n_MODEL_INPUTS = ("input_ids", "attention_mask", "token_type_ids")\n\n\ndef seed_everything(seed):\n    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef prepare_encoder(model_id, revision, asset_dir):\n    """Download an encoder once; store only its config and tokenizer assets.\n\n    The caller stores the trained encoder and head together in its checkpoint.\n    Transformers imports are lazy so CPU tests do not require that dependency.\n    """\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\n\n    if not isinstance(model_id, str) or not model_id.strip():\n        raise ValueError("model_id must be a nonempty string")\n    if not isinstance(revision, str) or not revision.strip():\n        raise ValueError("revision must be explicit")\n    config = AutoConfig.from_pretrained(model_id, revision=revision, trust_remote_code=False)\n    resolved = getattr(config, "_commit_hash", None)\n    if resolved is None and re.fullmatch(r"[0-9a-f]{40}", revision):\n        resolved = revision\n    if not isinstance(resolved, str) or not re.fullmatch(r"[0-9a-f]{40}", resolved):\n        raise RuntimeError("The immutable pretrained model revision was not resolved")\n    # Resolve a mutable branch exactly once, then pin every other asset to it.\n    options = {"revision": resolved, "trust_remote_code": False}\n    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, **options)\n    if not getattr(tokenizer, "is_fast", False):\n        raise ValueError("Complete overflow windows require a fast tokenizer")\n    encoder = AutoModel.from_pretrained(model_id, config=config, **options)\n    if hasattr(encoder.config, "use_cache"):\n        encoder.config.use_cache = False\n    if not getattr(encoder, "supports_gradient_checkpointing", False):\n        raise ValueError("The selected encoder must support gradient checkpointing")\n    encoder.gradient_checkpointing_enable(\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n    )\n    asset_dir = Path(asset_dir)\n    encoder_dir, tokenizer_dir = asset_dir / "encoder", asset_dir / "tokenizer"\n    encoder_dir.mkdir(parents=True, exist_ok=True)\n    tokenizer_dir.mkdir(parents=True, exist_ok=True)\n    encoder.config.save_pretrained(encoder_dir)\n    tokenizer.save_pretrained(tokenizer_dir)\n    return encoder, tokenizer, resolved\n\n\ndef _binary_targets(targets, count):\n    values = torch.as_tensor(targets)\n    if values.shape != (count, NUM_LABELS):\n        raise ValueError(f"targets must have shape ({count}, {NUM_LABELS})")\n    if not bool(torch.isfinite(values).all()) or not bool(((values == 0) | (values == 1)).all()):\n        raise ValueError("targets must contain only finite binary values")\n    return values.to(dtype=torch.long)\n\n\nclass EncodeDataset(Dataset):\n    """Retain all fast-tokenizer overflow windows, including the final tokens."""\n\n    def __init__(self, texts, targets, tokenizer, max_length=512, overlap=64):\n        texts = list(texts)\n        if any(not isinstance(text, str) for text in texts):\n            raise ValueError("Every dialogue must be a text string")\n        for name, value in (("max_length", max_length), ("overlap", overlap)):\n            if isinstance(value, bool) or not isinstance(value, int):\n                raise ValueError(f"{name} must be an integer")\n        usable = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n        if usable < 1 or not 0 <= overlap < usable:\n            raise ValueError("max_length must fit special tokens and 0 <= overlap < usable tokens")\n        if getattr(tokenizer, "pad_token_id", None) is None:\n            raise ValueError("The tokenizer must define a padding token")\n        separator = getattr(tokenizer, "sep_token", None)\n        if not isinstance(separator, str) or not separator:\n            raise ValueError("The tokenizer must define an utterance separator token")\n        self.tokenizer = tokenizer\n        self.targets = None if targets is None else _binary_targets(targets, len(texts)).clone()\n        self.documents = [[] for _ in texts]\n        for start in range(0, len(texts), 128):\n            dialogues = [re.sub(r"\\r\\n?|\\n", f" {separator} ", text)\n                         for text in texts[start:start + 128]]\n            encoded = tokenizer(\n                dialogues, truncation=True, max_length=max_length, stride=overlap,\n                return_overflowing_tokens=True, padding=False,\n                return_attention_mask=True,\n            )\n            if "overflow_to_sample_mapping" not in encoded:\n                raise ValueError("A fast tokenizer must return overflow_to_sample_mapping")\n            if "input_ids" not in encoded or "attention_mask" not in encoded:\n                raise ValueError("The tokenizer must return input_ids and attention_mask")\n            for index, source in enumerate(encoded["overflow_to_sample_mapping"]):\n                source = int(source)\n                if not 0 <= source < len(dialogues):\n                    raise ValueError("Tokenizer overflow mapping is out of range")\n                window = {key: list(encoded[key][index])\n                          for key in _MODEL_INPUTS if key in encoded}\n                length = len(window["input_ids"])\n                if not 0 < length <= max_length or any(len(value) != length for value in window.values()):\n                    raise ValueError("Tokenizer returned an invalid window length")\n                if not all(value in (0, 1) for value in window["attention_mask"]) or not any(window["attention_mask"]):\n                    raise ValueError("A real token window must have a binary, nonempty attention mask")\n                self.documents[start + source].append(window)\n        if any(not document for document in self.documents):\n            raise ValueError("Tokenizer returned a dialogue without any windows")\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        item = {"windows": self.documents[index]}\n        if self.targets is not None:\n            item["targets"] = self.targets[index]\n        return item\n\n    def collate(self, examples):\n        """Pad tokens and windows; keep one target vector for each dialogue."""\n        if not examples or any(not item["windows"] for item in examples):\n            raise ValueError("A batch must contain nonempty dialogue windows")\n        keys = set(examples[0]["windows"][0])\n        if any(set(window) != keys for item in examples for window in item["windows"]):\n            raise ValueError("Token windows must use the same model input fields")\n        if any(("targets" in item) != ("targets" in examples[0]) for item in examples):\n            raise ValueError("A batch cannot mix labeled and unlabeled dialogues")\n        batch_size = len(examples)\n        window_count = max(len(item["windows"]) for item in examples)\n        token_count = max(len(window["input_ids"]) for item in examples for window in item["windows"])\n        batch = {key: torch.full(\n            (batch_size, window_count, token_count),\n            self.tokenizer.pad_token_id if key == "input_ids" else 0,\n            dtype=torch.long,\n        ) for key in _MODEL_INPUTS if key in keys}\n        batch["window_mask"] = torch.zeros((batch_size, window_count), dtype=torch.bool)\n        for row, item in enumerate(examples):\n            for column, window in enumerate(item["windows"]):\n                length = len(window["input_ids"])\n                for key in keys:\n                    batch[key][row, column, :length] = torch.tensor(window[key], dtype=torch.long)\n                batch["window_mask"][row, column] = True\n        if "targets" in examples[0]:\n            batch["targets"] = torch.stack([item["targets"] for item in examples])\n        return batch\n\n\nclass DialogueClassifier(nn.Module):\n    """Train nine independent binary heads from all windows of each dialogue."""\n\n    def __init__(self, encoder, label_attention=False, contrastive=False, feature_dim=256):\n        super().__init__()\n        if isinstance(feature_dim, bool) or not isinstance(feature_dim, int) or feature_dim < 1:\n            raise ValueError("feature_dim must be a positive integer")\n        if not isinstance(label_attention, bool) or not isinstance(contrastive, bool):\n            raise ValueError("label_attention and contrastive must be boolean")\n        self.encoder = encoder\n        self.label_attention = label_attention\n        self.contrastive = contrastive\n        self.encoder_window_batch = 8\n        hidden_size = int(encoder.config.hidden_size)\n        self.projection = nn.Sequential(nn.Linear(hidden_size * 2, feature_dim), nn.GELU())\n        self.dropout = nn.Dropout(0.1)\n        if label_attention:\n            self.token_queries = nn.Parameter(torch.empty(NUM_LABELS, hidden_size))\n            self.window_queries = nn.Parameter(torch.empty(NUM_LABELS, feature_dim))\n            nn.init.normal_(self.token_queries, std=0.02)\n            nn.init.normal_(self.window_queries, std=0.02)\n        else:\n            self.register_parameter("token_queries", None)\n            self.register_parameter("window_queries", None)\n        self.class_weights = nn.Parameter(torch.empty(NUM_LABELS, feature_dim, 2))\n        self.class_bias = nn.Parameter(torch.zeros(NUM_LABELS, 2))\n        self.feature_weights = nn.Parameter(torch.empty(NUM_LABELS, feature_dim, CONTRASTIVE_DIM))\n        self.feature_bias = nn.Parameter(torch.zeros(NUM_LABELS, CONTRASTIVE_DIM))\n        nn.init.xavier_uniform_(self.class_weights)\n        nn.init.xavier_uniform_(self.feature_weights)\n\n    def forward(self, batch):\n        window_mask = batch.get("window_mask")\n        ids, token_mask = batch.get("input_ids"), batch.get("attention_mask")\n        if not isinstance(ids, torch.Tensor) or ids.ndim != 3:\n            raise ValueError("input_ids must have shape (batch, windows, tokens)")\n        if not isinstance(token_mask, torch.Tensor) or token_mask.shape != ids.shape:\n            raise ValueError("attention_mask must match input_ids")\n        if not isinstance(window_mask, torch.Tensor) or window_mask.shape != ids.shape[:2] or window_mask.dtype != torch.bool:\n            raise ValueError("window_mask must be a boolean (batch, windows) tensor")\n        if ids.shape[0] < 1 or not bool(window_mask.any(dim=1).all()):\n            raise ValueError("Every dialogue must contain at least one real window")\n        if not bool(((token_mask == 0) | (token_mask == 1)).all()):\n            raise ValueError("attention_mask must be binary")\n        if not bool((token_mask.sum(-1)[window_mask] > 0).all()):\n            raise ValueError("Real windows must contain attended tokens")\n        chunk_size = self.encoder_window_batch\n        if isinstance(chunk_size, bool) or not isinstance(chunk_size, int) or chunk_size < 1:\n            raise ValueError("encoder_window_batch must be a positive integer")\n        flat_mask = window_mask.flatten()\n        inputs = {}\n        for key in _MODEL_INPUTS:\n            if key in batch:\n                if batch[key].shape != ids.shape:\n                    raise ValueError(f"{key} must match input_ids")\n                inputs[key] = batch[key].flatten(0, 1)[flat_mask]\n        representations = []\n        for start in range(0, len(inputs["input_ids"]), chunk_size):\n            chunk = {key: value[start:start + chunk_size] for key, value in inputs.items()}\n            hidden = self.encoder(**chunk).last_hidden_state\n            if not bool(torch.isfinite(hidden).all()):\n                raise FloatingPointError("Encoder produced nonfinite hidden states")\n            if self.label_attention:\n                token_scores = torch.einsum("nth,lh->ntl", hidden.float(), self.token_queries.float())\n                token_scores = token_scores / math.sqrt(hidden.shape[-1])\n                token_scores = token_scores.masked_fill(~chunk["attention_mask"].bool().unsqueeze(-1), -torch.inf)\n                token_attention = torch.softmax(token_scores, dim=1).to(hidden.dtype)\n                context = torch.einsum("ntl,nth->nlh", token_attention, hidden)\n                cls = hidden[:, 0].unsqueeze(1).expand(-1, NUM_LABELS, -1)\n                representations.append(self.projection(torch.cat((cls, context), dim=-1)))\n            else:\n                mask = chunk["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n                mean = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)\n                representations.append(self.projection(torch.cat((hidden[:, 0], mean), dim=-1)))\n        valid = self.dropout(torch.cat(representations, dim=0))\n        if not bool(torch.isfinite(valid).all()):\n            raise FloatingPointError("Window projection produced nonfinite features")\n        padded_shape = (*window_mask.shape, *valid.shape[1:])\n        windows = valid.new_zeros(padded_shape)\n        windows = windows.flatten(0, 1).index_copy(\n            0, flat_mask.nonzero(as_tuple=False).flatten(), valid,\n        ).reshape(padded_shape)\n        if self.label_attention:\n            scores = (windows.float() * self.window_queries.float()[None, None]).sum(-1)\n            scores = scores / math.sqrt(windows.shape[-1])\n            scores = scores.masked_fill(~window_mask.unsqueeze(-1), -torch.inf)\n            attention = torch.softmax(scores, dim=1).to(windows.dtype)\n            pooled = (attention.unsqueeze(-1) * windows).sum(dim=1)\n        else:\n            pooled = windows.sum(dim=1) / window_mask.sum(dim=1, keepdim=True).to(windows.dtype)\n            pooled = pooled.unsqueeze(1).expand(-1, NUM_LABELS, -1)\n        pooled = self.dropout(pooled)\n        logits = torch.einsum("bld,ldc->blc", pooled, self.class_weights) + self.class_bias\n        features = torch.einsum("bld,ldf->blf", pooled, self.feature_weights) + self.feature_bias\n        features = F.normalize(features.float(), dim=-1)\n        if not bool(torch.isfinite(logits).all()) or not bool(torch.isfinite(features).all()):\n            raise FloatingPointError("Classifier produced nonfinite logits or features")\n        return logits, features\n\n\ndef predictions(logits):\n    """The learned head\'s winning class; no fitted post-training cutoff."""\n    if logits.ndim != 3 or logits.shape[1:] != (NUM_LABELS, 2):\n        raise ValueError("logits must have shape (batch, 9, 2)")\n    if not bool(torch.isfinite(logits).all()):\n        raise FloatingPointError("Cannot predict from nonfinite logits")\n    return logits.argmax(dim=-1)\n\n\ndef classification_loss(logits, targets, nausea_positive_weight=1.0):\n    """Weight only nausea-positive CE items, retaining the batch-by-label mean."""\n    if isinstance(nausea_positive_weight, bool) or not isinstance(nausea_positive_weight, Real):\n        raise ValueError("nausea_positive_weight must be a finite positive real number")\n    try:\n        positive_weight = float(nausea_positive_weight)\n    except (OverflowError, ValueError) as error:\n        raise ValueError("nausea_positive_weight must be a finite positive real number") from error\n    if not math.isfinite(positive_weight) or positive_weight <= 0:\n        raise ValueError("nausea_positive_weight must be a finite positive real number")\n    predictions(logits)\n    targets = _binary_targets(targets, len(logits)).to(logits.device)\n    if len(logits) < 1:\n        raise ValueError("classification_loss requires a nonempty batch")\n    if positive_weight == 1.0:\n        loss = F.cross_entropy(logits.float().reshape(-1, 2), targets.reshape(-1))\n    else:\n        per_item = F.cross_entropy(\n            logits.float().reshape(-1, 2), targets.reshape(-1), reduction="none",\n        ).reshape(-1, NUM_LABELS)\n        weights = torch.ones_like(per_item)\n        weights[:, 6] = torch.where(targets[:, 6] == 1, positive_weight, 1.0)\n        loss = (per_item * weights).mean()\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Classification loss is nonfinite")\n    return loss\n\n\ndef binary_supervised_contrastive_loss(features1, features2, targets, temperature=0.1):\n    """Binary supervised contrastive loss per symptom, with balanced anchors.\n\n    The two views come from independent dropout forwards of the same dialogues.\n    Same-class examples are positives, self matches are excluded, and each\n    present binary class receives equal anchor weight within each symptom.\n    Thus the many symptom-negative examples cannot dominate the anchor average.\n    """\n    if features1.ndim != 3 or features1.shape[1] != NUM_LABELS or features1.shape != features2.shape:\n        raise ValueError("Both feature views must have matching (batch, 9, feature_dim) shapes")\n    if features1.shape[0] < 1 or features1.shape[2] < 1:\n        raise ValueError("Contrastive loss requires nonempty feature views")\n    if not bool(torch.isfinite(features1).all()) or not bool(torch.isfinite(features2).all()):\n        raise FloatingPointError("Contrastive features must be finite")\n    if isinstance(temperature, bool) or not math.isfinite(float(temperature)) or temperature <= 0:\n        raise ValueError("temperature must be positive and finite")\n    labels = _binary_targets(targets, len(features1)).to(features1.device)\n    features = F.normalize(torch.cat((features1.float(), features2.float()), dim=0), dim=-1)\n    labels = torch.cat((labels, labels), dim=0)\n    self_mask = torch.eye(len(features), dtype=torch.bool, device=features.device)\n    losses = []\n    for label in range(NUM_LABELS):\n        scores = features[:, label] @ features[:, label].T / float(temperature)\n        denominator = torch.logsumexp(scores.masked_fill(self_mask, -torch.inf), dim=1, keepdim=True)\n        log_probability = scores - denominator\n        positive = (labels[:, label, None] == labels[None, :, label]) & ~self_mask\n        anchor_loss = -(log_probability * positive).sum(1) / positive.sum(1)\n        class_losses = [anchor_loss[labels[:, label] == value].mean()\n                        for value in (0, 1) if bool((labels[:, label] == value).any())]\n        losses.append(torch.stack(class_losses).mean())\n    loss = torch.stack(losses).mean()\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Supervised contrastive loss is nonfinite")\n    return loss\n', 'audit.py': '"""Training-only review candidates without transcript or source identifier export."""\nfrom __future__ import annotations\n\nimport csv\nfrom datetime import datetime, timezone\nimport json\nfrom pathlib import Path\nimport re\nfrom uuid import uuid4\n\nimport numpy as np\n\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\nSTRONG_CUE_PATTERNS = {\n    "nausea_spellings": r"메스꺼|메스껍|메슥|매스꺼|매스껍|매슥",\n    "fluttering": r"울렁",\n    "retching": r"구역질",\n    "colloquial_nausea": r"미식|느글",\n    "prospective_vomiting": r"토할\\s*(?:것|거|꺼)\\s*같|토할\\s*(?:라고|려고)|토가\\s*나올",\n}\nQUESTION_PATTERN = r"\\?|나요|까요|습니까|신가|세요"\nNEGATION_PATTERN = r"아니|없|않|못|안\\s|아닌"\nREVIEW_STATUS = "unreviewed_candidate"\nCSV_FIELDS = (\n    "original_index", "split", "labels", "cue_groups", "candidate_reason",\n    "has_question_expression", "has_negation_expression",\n    "has_nonquestion_nonnegation_cue", "review_status", "review_note",\n)\n\n\ndef _split_indices(values, rows: int, name: str) -> np.ndarray:\n    indices = np.asarray(values)\n    if indices.ndim != 1 or indices.dtype.kind not in "iu":\n        raise ValueError(f"{name} must be a one-dimensional integer index array")\n    if np.any(indices < 0) or np.any(indices >= rows):\n        raise ValueError(f"{name} contains an out-of-range index")\n    if len(np.unique(indices)) != len(indices):\n        raise ValueError(f"{name} contains duplicate indices")\n    return indices.astype(np.int64, copy=False)\n\n\ndef write_audit(paths, texts, targets, fit, holdout, output_dir) -> dict:\n    """Audit a caller-supplied Training partition without changing its gold targets.\n\n    ``paths`` is used only to validate row alignment; files and filenames are never\n    read or exported. Callers must supply Training rows, never official Validation.\n    Regex flags identify review candidates, not symptom presence or label errors.\n    Each call creates a new child directory and cannot overwrite an earlier audit.\n    """\n    texts = list(texts)\n    rows = len(texts)\n    if not rows or len(paths) != rows or any(not isinstance(text, str) for text in texts):\n        raise ValueError("paths and string texts must align and contain at least one row")\n    gold = np.asarray(targets)\n    if (gold.shape != (rows, len(TARGET_SYMPTOMS)) or gold.dtype.kind not in "biu"\n            or not np.isin(gold, [0, 1]).all()):\n        raise ValueError("targets must be a binary array with shape (rows, 9)")\n    fit = _split_indices(fit, rows, "fit")\n    holdout = _split_indices(holdout, rows, "holdout")\n    if np.intersect1d(fit, holdout).size:\n        raise ValueError("fit and holdout must be disjoint")\n    if not np.array_equal(np.sort(np.concatenate((fit, holdout))), np.arange(rows)):\n        raise ValueError("fit and holdout must partition every Training row")\n\n    cue_regexes = {name: re.compile(pattern) for name, pattern in STRONG_CUE_PATTERNS.items()}\n    question_regex, negation_regex = re.compile(QUESTION_PATTERN), re.compile(NEGATION_PATTERN)\n    cue_flags = np.zeros((rows, len(cue_regexes)), dtype=bool)\n    unqualified_flags = np.zeros_like(cue_flags)\n    questions, negations = np.zeros(rows, dtype=bool), np.zeros(rows, dtype=bool)\n    for index, text in enumerate(texts):\n        lines = text.splitlines()\n        questions[index] = bool(question_regex.search(text))\n        negations[index] = bool(negation_regex.search(text))\n        for column, regex in enumerate(cue_regexes.values()):\n            cue_flags[index, column] = any(regex.search(line) for line in lines)\n            unqualified_flags[index, column] = any(\n                regex.search(line) and not question_regex.search(line) and not negation_regex.search(line)\n                for line in lines\n            )\n\n    nausea = gold[:, TARGET_SYMPTOMS.index("오심")].astype(bool)\n    cue, unqualified = cue_flags.any(axis=1), unqualified_flags.any(axis=1)\n    candidates = cue | nausea\n    split_name = np.empty(rows, dtype=object)\n    split_name[fit], split_name[holdout] = "fit", "holdout"\n\n    def counts(indices):\n        local_nausea, local_cue = nausea[indices], cue[indices]\n        local_unqualified = unqualified[indices]\n        return {\n            "samples": int(len(indices)),\n            "nausea_positive": int(local_nausea.sum()),\n            "strong_cue_samples": int(local_cue.sum()),\n            "strong_cue_label_positive": int((local_cue & local_nausea).sum()),\n            "strong_cue_label_negative": int((local_cue & ~local_nausea).sum()),\n            "no_strong_cue_label_positive": int((~local_cue & local_nausea).sum()),\n            "nonquestion_nonnegation_cue_samples": int(local_unqualified.sum()),\n            "nonquestion_nonnegation_cue_label_positive": int((local_unqualified & local_nausea).sum()),\n            "nonquestion_nonnegation_cue_label_negative": int((local_unqualified & ~local_nausea).sum()),\n            "review_candidates": int(candidates[indices].sum()),\n            "cue_groups": {\n                name: {\n                    "samples": int(cue_flags[indices, column].sum()),\n                    "label_positive": int((cue_flags[indices, column] & local_nausea).sum()),\n                    "label_negative": int((cue_flags[indices, column] & ~local_nausea).sum()),\n                    "nonquestion_nonnegation_samples": int(unqualified_flags[indices, column].sum()),\n                    "nonquestion_nonnegation_label_positive": int(\n                        (unqualified_flags[indices, column] & local_nausea).sum()),\n                    "nonquestion_nonnegation_label_negative": int(\n                        (unqualified_flags[indices, column] & ~local_nausea).sum()),\n                }\n                for column, name in enumerate(cue_regexes)\n            },\n        }\n\n    summary = {\n        "format": "mission3_training_nausea_audit_v1",\n        "source_policy": "Training only; official Validation must not be supplied",\n        "review_status": REVIEW_STATUS,\n        "targets_changed": False,\n        "rows_removed": 0,\n        "index_convention": "0-based original_index in the caller\'s sorted Training paths",\n        "strong_cue_patterns": STRONG_CUE_PATTERNS.copy(),\n        "question_pattern": QUESTION_PATTERN,\n        "negation_pattern": NEGATION_PATTERN,\n        "interpretation": (\n            "Regex matches and nonquestion/nonnegation flags are review candidates, not human semantic "\n            "judgments. A match does not establish nausea, and absence does not establish a label error. "\n            "Question/negation matches can concern another symptom; \'not feeling well\' can describe a "\n            "positive symptom. Gold targets are preserved, with no automatic relabeling or removal."\n        ),\n        "counts": {"training": counts(np.arange(rows)), "fit": counts(fit), "holdout": counts(holdout)},\n    }\n    parent = Path(output_dir)\n    child = "audit_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex\n    run_dir = parent / child\n    run_dir.mkdir(parents=True, exist_ok=False)\n    summary["output_dir"] = str(run_dir.resolve())\n    summary["files"] = {"summary": "audit_summary.json", "candidates": "review_candidates.csv"}\n    with (run_dir / "review_candidates.csv").open("x", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=CSV_FIELDS)\n        writer.writeheader()\n        for index in np.flatnonzero(candidates):\n            reason = ("strong_cue_label_positive" if nausea[index] else "strong_cue_label_negative")\n            if not cue[index]:\n                reason = "no_strong_cue_label_positive"\n            writer.writerow({\n                "original_index": int(index), "split": split_name[index],\n                "labels": json.dumps([name for name, flag in zip(TARGET_SYMPTOMS, gold[index]) if flag],\n                                     ensure_ascii=False),\n                "cue_groups": json.dumps([name for column, name in enumerate(cue_regexes)\n                                          if cue_flags[index, column]]),\n                "candidate_reason": reason,\n                "has_question_expression": int(questions[index]),\n                "has_negation_expression": int(negations[index]),\n                "has_nonquestion_nonnegation_cue": int(unqualified[index]),\n                "review_status": REVIEW_STATUS, "review_note": "",\n            })\n    with (run_dir / "audit_summary.json").open("x", encoding="utf-8") as handle:\n        json.dump(summary, handle, ensure_ascii=False, indent=2)\n        handle.write("\\n")\n    return summary\n', 'runtime.py': '"""Training-only model selection and categorical Mission 3 inference."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport time\nimport uuid\n\nimport joblib\nimport numpy as np\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.pipeline import FeatureUnion\nfrom sklearn.multiclass import OneVsRestClassifier\nfrom sklearn.naive_bayes import ComplementNB\nfrom sklearn.svm import LinearSVC\n\nfrom . import data\nfrom .bootstrap import prepare_data\n\n\ndef dataset_digest(texts, targets):\n    digest = hashlib.sha256()\n    for text, target in zip(texts, targets):\n        digest.update(text.encode("utf-8"))\n        digest.update(b"\\0")\n        digest.update(np.asarray(target, dtype=np.int8).tobytes())\n    return digest.hexdigest()\n\n\ndef feature_extractor(settings):\n    return FeatureUnion([\n        ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5),\n                                min_df=2, max_features=settings.get("char_features", 80000),\n                                sublinear_tf=True, dtype=np.float32)),\n        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2,\n                                max_features=settings.get("word_features", 20000),\n                                sublinear_tf=True, dtype=np.float32)),\n    ])\n\n\ndef sparse_estimator(name, seed):\n    # These are standard categorical estimators; no score cutoff is searched.\n    if name == "svm_c1":\n        base = LinearSVC(C=1.0, dual="auto", max_iter=5000, random_state=seed)\n    elif name == "svm_c4":\n        base = LinearSVC(C=4.0, dual="auto", max_iter=5000, random_state=seed)\n    elif name == "complement_nb":\n        base = ComplementNB(alpha=1.0)\n    else:\n        raise ValueError(f"Unknown sparse candidate: {name}")\n    return OneVsRestClassifier(base, n_jobs=1)\n\n\ndef train_sparse(paths, texts, targets, fit, holdout, local_dir, output_dir, settings):\n    vectorizer = feature_extractor(settings)\n    train_x = vectorizer.fit_transform([texts[i] for i in fit])\n    holdout_x = vectorizer.transform([texts[i] for i in holdout])\n    candidates = []\n    best = None\n    for name in settings.get("sparse_candidates", ["svm_c1", "svm_c4", "complement_nb"]):\n        started = time.monotonic()\n        estimator = sparse_estimator(name, settings["seed"])\n        estimator.fit(train_x, targets[fit])\n        predictions = estimator.predict(holdout_x).astype(np.int8)\n        report = data.score_predictions(targets[holdout], predictions)\n        candidates.append({"name": name, "seconds": time.monotonic() - started, **report})\n        data.write_json(output_dir / "candidate_metrics.json", candidates)\n        print(f"Training holdout {name}: Macro F1={report[\'macro_f1\']:.6f}", flush=True)\n        if best is None or report["macro_f1"] > best[0]:\n            best = (report["macro_f1"], name, predictions, report)\n    if best is None:\n        raise ValueError("At least one sparse candidate is required")\n    np.savez_compressed(output_dir / "holdout_predictions.npz", indices=holdout,\n                        targets=targets[holdout], predictions=best[2])\n    data.export_errors([paths[i] for i in holdout],\n                       [texts[i] for i in holdout], targets[holdout], best[2],\n                       output_dir / "training_errors")\n    del train_x, holdout_x, estimator, vectorizer\n    gc.collect()\n    print(f"Selected using Training only: {best[1]}; refitting all Training", flush=True)\n    vectorizer = feature_extractor(settings)\n    full_x = vectorizer.fit_transform(texts)\n    estimator = sparse_estimator(best[1], settings["seed"])\n    estimator.fit(full_x, targets)\n    joblib.dump({"vectorizer": vectorizer, "estimator": estimator}, local_dir / "sparse.joblib", compress=3)\n    shutil.copy2(local_dir / "sparse.joblib", output_dir / "sparse.joblib")\n    manifest = {"kind": "sparse", "artifact": "sparse.joblib", "selected_name": best[1]}\n    return manifest, best[3]\n\n\ndef _save_weights(model, path):\n    import torch\n    temporary = path.with_name(path.name + ".tmp")\n    # CPU tensors make the saved checkpoint device-independent.\n    torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, temporary)\n    temporary.replace(path)\n\n\ndef _load_weights(model, path):\n    import torch\n    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)\n\n\ndef _loader(texts, targets, tokenizer, settings, shuffle):\n    from torch.utils.data import DataLoader\n    from .neural import EncodeDataset\n    dataset = EncodeDataset(texts, targets, tokenizer,\n                            max_length=settings["max_length"], overlap=settings["overlap"])\n    return DataLoader(dataset, batch_size=settings["batch_size"], shuffle=shuffle,\n                      num_workers=0, collate_fn=dataset.collate)\n\n\ndef _to_device(batch, device):\n    return {key: value.to(device) for key, value in batch.items()}\n\n\ndef _predict(model, loader, device):\n    import torch\n    from .neural import predictions\n    result = []\n    model.eval()\n    with torch.inference_mode():\n        for batch in loader:\n            logits, _ = model(_to_device(batch, device))\n            result.append(predictions(logits).cpu().numpy())\n    if not result:\n        raise ValueError("Cannot predict an empty dataset")\n    return np.concatenate(result).astype(np.int8)\n\n\ndef _fit_epochs(model, loader, settings, device, output_dir, prefix,\n                holdout_loader=None, holdout_targets=None, fixed_epochs=None):\n    import torch\n    from .neural import classification_loss, binary_supervised_contrastive_loss\n    encoder_parameters = list(model.encoder.parameters())\n    encoder_ids = {id(parameter) for parameter in encoder_parameters}\n    head_parameters = [p for p in model.parameters() if id(p) not in encoder_ids]\n    optimizer = torch.optim.AdamW([\n        {"params": encoder_parameters, "lr": settings["learning_rate"]},\n        {"params": head_parameters, "lr": settings["head_learning_rate"]},\n    ], weight_decay=0.01)\n    epochs = fixed_epochs or settings["max_epochs"]\n    schedule_epochs = settings["max_epochs"]\n    if type(epochs) is not int or not 1 <= epochs <= schedule_epochs:\n        raise ValueError("Executed epochs must fit within the selection schedule horizon")\n    accumulation = settings["gradient_accumulation"]\n    updates_per_epoch = (len(loader) + accumulation - 1) // accumulation\n    # Preserve the selected epoch\'s relative LR trajectory during fresh full refit.\n    total_updates = updates_per_epoch * schedule_epochs\n    warmup = max(1, int(total_updates * 0.1))\n\n    def schedule(step):\n        if step < warmup:\n            return (step + 1) / warmup\n        return max(0.0, (total_updates - step) / max(1, total_updates - warmup))\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\n    use_cuda = device.type == "cuda"\n    ce_views = settings.get("ce_views", 2 if settings["contrastive_weight"] else 1)\n    if type(ce_views) is not int or ce_views not in (1, 2) or (settings["contrastive_weight"] and ce_views != 2):\n        raise ValueError("ce_views must be 1 or 2, and contrastive training requires 2")\n    dtype = torch.bfloat16 if use_cuda and torch.cuda.is_bf16_supported() else torch.float16\n    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda and dtype == torch.float16)\n    best_f1, best_epoch, stale = -1.0, 0, 0\n    history = []\n    weight_path = output_dir / f"{prefix}_weights.pt"\n    for epoch in range(1, epochs + 1):\n        started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        total_loss, total_ce, total_contrastive = 0.0, 0.0, 0.0\n        nausea_batch_counts = {}\n        for step, batch in enumerate(loader):\n            batch = _to_device(batch, device)\n            positive_count = int(batch["targets"][:, 6].sum())\n            nausea_batch_counts[str(positive_count)] = nausea_batch_counts.get(str(positive_count), 0) + 1\n            with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_cuda):\n                logits, features1 = model(batch)\n                ce = classification_loss(logits, batch["targets"],\n                                         nausea_positive_weight=settings.get("nausea_positive_weight", 1.0))\n                contrastive = logits.new_zeros(())\n                if ce_views == 2:\n                    logits2, features2 = model(batch)\n                    ce2 = classification_loss(logits2, batch["targets"],\n                                              nausea_positive_weight=settings.get("nausea_positive_weight", 1.0))\n                    ce = (ce + ce2) / 2\n                if settings["contrastive_weight"]:\n                    contrastive = binary_supervised_contrastive_loss(features1, features2, batch["targets"])\n                loss = ce + settings["contrastive_weight"] * contrastive\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Non-finite training loss; no checkpoint was accepted")\n            group_start = (step // accumulation) * accumulation\n            group_size = min(accumulation, len(loader) - group_start)\n            scaler.scale(loss / group_size).backward()\n            total_loss += float(loss.detach())\n            total_ce += float(ce.detach())\n            total_contrastive += float(contrastive.detach())\n            if (step + 1) % accumulation == 0 or step + 1 == len(loader):\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)\n                scaler.step(optimizer)\n                scaler.update()\n                scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            if (step + 1) % settings.get("log_every", 250) == 0:\n                print(f"{prefix} epoch {epoch}: {step + 1}/{len(loader)} batches", flush=True)\n        row = {"epoch": epoch, "loss": total_loss / len(loader),\n               "scheduler_horizon_epochs": schedule_epochs, "ce_views": ce_views,\n               "classification_loss": total_ce / len(loader),\n               "contrastive_loss": total_contrastive / len(loader),\n               "nausea_positive_count_per_batch": nausea_batch_counts,\n               "seconds": time.monotonic() - started}\n        if holdout_loader is not None:\n            predicted = _predict(model, holdout_loader, device)\n            metrics = data.score_predictions(holdout_targets, predicted)\n            row["holdout_macro_f1"] = metrics["macro_f1"]\n            row["holdout_metrics"] = metrics\n            if metrics["macro_f1"] > best_f1:\n                best_f1, best_epoch, stale = metrics["macro_f1"], epoch, 0\n                _save_weights(model, weight_path)\n                data.write_json(output_dir / "selected_holdout_metrics.json", metrics)\n            else:\n                stale += 1\n            print(f"Training holdout epoch {epoch}: Macro F1={metrics[\'macro_f1\']:.6f}", flush=True)\n        else:\n            _save_weights(model, weight_path)\n            best_epoch = epoch\n            print(f"All-Training refit epoch {epoch}/{epochs}: loss={row[\'loss\']:.5f}", flush=True)\n        history.append(row)\n        data.write_json(output_dir / f"{prefix}_history.json", history)\n        if holdout_loader is not None and stale >= settings["patience"]:\n            break\n    return best_epoch, weight_path\n\n\ndef train_neural(paths, texts, targets, fit, holdout, local_dir, output_dir, settings):\n    import torch\n    from .neural import DialogueClassifier, prepare_encoder, seed_everything\n    if not torch.cuda.is_available() and not settings.get("allow_cpu", False):\n        raise RuntimeError("This version needs a Colab GPU runtime; ver10 runs on CPU")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    seed_everything(settings["seed"])\n    asset_dir = output_dir / "assets"\n    encoder, tokenizer, revision = prepare_encoder(settings["model_id"], settings["revision"], asset_dir)\n    settings = {**settings, "resolved_revision": revision}\n    data.write_json(output_dir / "settings.json", settings)\n    model = DialogueClassifier(encoder, label_attention=settings["label_attention"],\n                               contrastive=bool(settings["contrastive_weight"]))\n    model.encoder_window_batch = settings["encoder_window_batch"]\n    model.to(device)\n    fit_loader = _loader([texts[i] for i in fit], targets[fit], tokenizer, settings, True)\n    holdout_loader = _loader([texts[i] for i in holdout], targets[holdout], tokenizer, settings, False)\n    best_epoch, selected_path = _fit_epochs(model, fit_loader, settings, device, output_dir,\n                                           "selection", holdout_loader, targets[holdout])\n    _load_weights(model, selected_path)\n    predicted = _predict(model, holdout_loader, device)\n    metrics = data.score_predictions(targets[holdout], predicted)\n    np.savez_compressed(output_dir / "holdout_predictions.npz", indices=holdout,\n                        targets=targets[holdout], predictions=predicted)\n    data.export_errors([paths[i] for i in holdout], [texts[i] for i in holdout],\n                       targets[holdout], predicted, output_dir / "training_errors")\n    if settings.get("refit_full_training", True):\n        del model, encoder, fit_loader, holdout_loader\n        gc.collect()\n        if device.type == "cuda":\n            torch.cuda.empty_cache()\n        seed_everything(settings["seed"])\n        # Fresh public encoder; selected weights do not seed the refit.\n        encoder, tokenizer, _ = prepare_encoder(settings["model_id"], revision, asset_dir)\n        model = DialogueClassifier(encoder, label_attention=settings["label_attention"],\n                                   contrastive=bool(settings["contrastive_weight"]))\n        model.encoder_window_batch = settings["encoder_window_batch"]\n        model.to(device)\n        full_loader = _loader(texts, targets, tokenizer, settings, True)\n        _, final_weights = _fit_epochs(model, full_loader, settings, device, output_dir,\n                                       "final", fixed_epochs=best_epoch)\n        fitted_samples = len(targets)\n    else:\n        final_weights = selected_path\n        fitted_samples = len(fit)\n    # A final artifact exists only once its declared refit has completed.\n    manifest = {"kind": "neural", "weights": final_weights.name, "assets": "assets",\n                "model_id": settings["model_id"], "resolved_revision": revision,\n                "selected_epochs": best_epoch, "fitted_samples": fitted_samples,\n                "label_attention": settings["label_attention"],\n                "contrastive": bool(settings["contrastive_weight"]),\n                "max_length": settings["max_length"], "overlap": settings["overlap"],\n                "batch_size": settings["batch_size"], "encoder_window_batch": settings["encoder_window_batch"]}\n    del model, encoder\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return manifest, metrics\n\n\ndef refit_selected(texts, targets, settings, selected_epochs, output_dir):\n    """Refit the chosen candidate once from its public encoder, using Training only."""\n    import torch\n    from .neural import DialogueClassifier, prepare_encoder, seed_everything\n    if type(selected_epochs) is not int or selected_epochs < 1:\n        raise ValueError("selected_epochs must be a positive integer")\n    if not torch.cuda.is_available() and not settings.get("allow_cpu", False):\n        raise RuntimeError("ver14 requires a GPU runtime for pretrained training")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    seed_everything(settings["seed"])\n    encoder, tokenizer, revision = prepare_encoder(settings["model_id"], settings["revision"], output_dir / "assets")\n    model = DialogueClassifier(encoder, label_attention=settings["label_attention"],\n                               contrastive=bool(settings["contrastive_weight"]))\n    model.encoder_window_batch = settings["encoder_window_batch"]\n    model.to(device)\n    loader = _loader(texts, targets, tokenizer, settings, True)\n    _, weights = _fit_epochs(model, loader, settings, device, output_dir, "final", fixed_epochs=selected_epochs)\n    _load_weights(model, weights)\n    probe_texts = texts[:min(12, len(texts))]\n    probe_predictions = _predict(model, _loader(probe_texts, None, tokenizer, settings, False), device)\n    manifest = {"kind": "neural", "weights": weights.name, "assets": "assets",\n                "model_id": settings["model_id"], "resolved_revision": revision,\n                "selected_epochs": selected_epochs, "fitted_samples": len(targets),\n                "label_attention": settings["label_attention"], "contrastive": bool(settings["contrastive_weight"]),\n                "nausea_positive_weight": settings["nausea_positive_weight"],\n                "max_length": settings["max_length"], "overlap": settings["overlap"],\n                "batch_size": settings["batch_size"], "encoder_window_batch": settings["encoder_window_batch"]}\n    del model, encoder, loader\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return manifest, probe_predictions\n\n\ndef _relative_artifact(root, relative):\n    path = (root / relative).resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError("Model artifact path escapes checkpoint directory")\n    return path\n\n\ndef predict_checkpoint(checkpoint, texts):\n    checkpoint = Path(checkpoint)\n    manifest = json.loads(checkpoint.read_text(encoding="utf-8-sig"))\n    if manifest.get("format") != "mission3_categorical_v1" or tuple(manifest["labels"]) != data.TARGET_SYMPTOMS:\n        raise ValueError("Unsupported checkpoint or symptom ordering")\n    root = checkpoint.parent\n    if manifest["kind"] == "sparse":\n        bundle = joblib.load(_relative_artifact(root, manifest["artifact"]))\n        return bundle["estimator"].predict(bundle["vectorizer"].transform(texts)).astype(np.int8)\n    if manifest["kind"] != "neural":\n        raise ValueError("Unknown checkpoint model kind")\n    import torch\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\n    from .neural import DialogueClassifier\n    assets = _relative_artifact(root, manifest["assets"])\n    config = AutoConfig.from_pretrained(assets / "encoder", local_files_only=True, trust_remote_code=False)\n    encoder = AutoModel.from_config(config, trust_remote_code=False)\n    tokenizer = AutoTokenizer.from_pretrained(assets / "tokenizer", local_files_only=True, trust_remote_code=False)\n    model = DialogueClassifier(encoder, label_attention=manifest["label_attention"],\n                               contrastive=manifest["contrastive"])\n    model.encoder_window_batch = manifest["encoder_window_batch"]\n    _load_weights(model, _relative_artifact(root, manifest["weights"]))\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model.to(device)\n    loader = _loader(texts, None, tokenizer, manifest, False)\n    return _predict(model, loader, device)\n\n\ndef inference(checkpoint, label_dir, output):\n    paths, texts, _ = data.load_dataset(label_dir, include_targets=False)\n    predicted = predict_checkpoint(checkpoint, texts)\n    data.write_predictions(paths, predicted, output)\n    return paths, predicted\n\n\ndef run(settings, runtime_sources=None):\n    started = time.monotonic()\n    settings = dict(settings)\n    train_dir, validation_dir = prepare_data(settings["input_zip"], settings["work_dir"],\n                                             settings.get("train_dir", ""), settings.get("validation_dir", ""))\n    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n    output_dir = Path(settings["output_dir"]) / run_id\n    output_dir.mkdir(parents=True, exist_ok=False)\n    local_dir = Path(settings["work_dir"]) / run_id\n    local_dir.mkdir(parents=True, exist_ok=False)\n    data.write_json(output_dir / "settings.json", settings)\n    if runtime_sources:\n        submission = output_dir / "submission"\n        for name, source in runtime_sources.items():\n            if name.startswith("test_") or name == "build_notebooks.py":\n                continue\n            target = submission / name\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_text(source, encoding="utf-8")\n    paths, texts, targets = data.load_dataset(train_dir, include_targets=True)\n    fit, holdout = data.make_split(texts, targets, settings["holdout_size"], settings["seed"])\n    data.write_json(output_dir / "training_diagnostics.json", data.training_diagnostics(texts, targets))\n    np.savez_compressed(output_dir / "training_split.npz", fit=fit, holdout=holdout)\n    data.write_json(output_dir / "run_identity.json", {\n        "version": settings["version"], "training_digest": dataset_digest(texts, targets),\n        "training_samples": len(texts), "fit_samples": len(fit), "holdout_samples": len(holdout),\n        "model_selection": "Training internal holdout only", "decision_rule": "categorical predict/argmax",\n    })\n    print(f"Training {len(texts):,}; fit {len(fit):,}; holdout {len(holdout):,}", flush=True)\n    print("Training label counts:", dict(zip(data.TARGET_SYMPTOMS, targets.sum(axis=0).tolist())), flush=True)\n    try:\n        trainer = train_sparse if settings["backend"] == "sparse" else train_neural\n        manifest, metrics = trainer(paths, texts, targets, fit, holdout, local_dir, output_dir, settings)\n        manifest.update(format="mission3_categorical_v1", labels=list(data.TARGET_SYMPTOMS),\n                        version=settings["version"], decision_rule="categorical predict/argmax")\n        checkpoint = output_dir / f"mission3_{settings[\'version\']}_model.ckpt"\n        data.write_json(output_dir / "training_metrics.json", metrics)\n        data.write_json(checkpoint, manifest)\n        # Reopen the serialized model on the holdout text before official inference.\n        probe = [texts[i] for i in holdout[:min(12, len(holdout))]]\n        roundtrip = predict_checkpoint(checkpoint, probe)\n        if roundtrip.shape != (len(probe), len(data.TARGET_SYMPTOMS)):\n            raise ValueError("Checkpoint roundtrip shape mismatch")\n        if settings.get("run_validation", True):\n            validation_paths, predicted = inference(checkpoint, validation_dir, output_dir / "mission3.csv")\n            # Official targets are accessed only after predictions are saved.\n            scored_paths, _, validation_targets = data.load_dataset(validation_dir, include_targets=True)\n            if [p.name for p in scored_paths] != [p.name for p in validation_paths]:\n                raise ValueError("Official evaluation file order mismatch")\n            report = data.score_predictions(validation_targets, predicted)\n            data.write_json(output_dir / "validation_metrics.json", report)\n            print(f"Official Validation (final diagnostic): Macro F1={report[\'macro_f1\']:.6f}", flush=True)\n            for name, value in report["per_symptom"].items():\n                print(f"  {name}: F1={value[\'f1\']:.4f}, P={value[\'precision\']:.4f}, R={value[\'recall\']:.4f}", flush=True)\n        data.write_json(output_dir / "runtime.json", {"status": "complete", "seconds": time.monotonic() - started})\n    except Exception as error:\n        data.write_json(output_dir / "runtime.json", {"status": "failed", "seconds": time.monotonic() - started,\n                                                      "error_type": type(error).__name__})\n        raise\n    print("Saved results:", output_dir, flush=True)\n    return output_dir\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--audio_dir", required=True)  # Official interface; Mission 3 uses text only.\n    parser.add_argument("--label_dir", required=True)\n    parser.add_argument("--ckpt_path", required=True)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args()\n    inference(args.ckpt_path, args.label_dir, args.output)\n\n\nif __name__ == "__main__":\n    main()\n', 'runtime_suite.py': '"""Ordered Training audit, weighted CE trials, and matched contrastive ablations."""\nfrom __future__ import annotations\n\nimport csv\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport math\nfrom numbers import Real\nfrom pathlib import Path\nimport platform\nimport shutil\nimport time\nimport uuid\n\nimport numpy as np\n\nfrom . import data, runtime\nfrom .audit import write_audit\n\n\nNAUSEA = data.TARGET_SYMPTOMS[6]\nPOLICY = "Training Macro F1 first, with no regression in baseline Macro F1 or nausea F1"\nBASELINE = "stage2_ce_w1"\n\n\ndef _stamp():\n    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n\n\ndef _digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                     allow_nan=False).encode("utf-8")).hexdigest()\n\n\ndef _read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8-sig"))\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _safe(root, relative):\n    candidate = root / relative\n    path = candidate.resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError("Saved artifact path escapes its run directory")\n    if any(part.is_symlink() for part in (candidate, *candidate.parents) if part != root.parent):\n        raise ValueError("Symlink artifacts cannot be resumed")\n    return path\n\n\ndef _hashes(root, manifest, extra=()):\n    weights = _safe(root, manifest["weights"])\n    assets = _safe(root, manifest["assets"])\n    if not weights.is_file() or not assets.is_dir():\n        raise FileNotFoundError("Completed model is missing weights or assets")\n    files = [weights, *sorted(p for p in assets.rglob("*") if p.is_file()),\n             *(_safe(root, name) for name in extra)]\n    if not any(path.parent.name == "tokenizer" for path in files):\n        raise FileNotFoundError("Completed model has no tokenizer assets")\n    return {str(path.relative_to(root.resolve())).replace("\\\\", "/"): _hash(path) for path in files}\n\n\ndef _verify_hashes(root, hashes):\n    if not hashes:\n        raise ValueError("A completion record must contain artifact hashes")\n    for name, expected in hashes.items():\n        path = _safe(root, name)\n        if not path.is_file() or _hash(path) != expected:\n            raise ValueError(f"Saved artifact is missing or changed: {name}")\n\n\ndef validate_settings(settings):\n    settings = dict(settings)\n    if settings.get("version") != "ver14" or settings.get("backend") != "neural":\n        raise ValueError("This runner supports ver14 neural experiments only")\n    weights = settings.get("nausea_weight_candidates", [1.0, 2.0, 3.0])\n    if (not isinstance(weights, list) or not weights or weights[0] != 1.0\n            or any(isinstance(w, bool) or not isinstance(w, Real) or not math.isfinite(w) or w < 1 for w in weights)\n            or len(weights) != len(set(weights))):\n        raise ValueError("nausea_weight_candidates must be distinct finite weights >=1, starting with 1")\n    settings["nausea_weight_candidates"] = [float(w) for w in weights]\n    for key in ("batch_size", "gradient_accumulation", "stage3_batch_size", "stage3_gradient_accumulation",\n                "max_epochs", "patience", "encoder_window_batch", "max_length", "log_every"):\n        if type(settings.get(key)) is not int or settings[key] < 1:\n            raise ValueError(f"{key} must be a positive integer")\n    if settings["batch_size"] * settings["gradient_accumulation"] != (\n            settings["stage3_batch_size"] * settings["stage3_gradient_accumulation"]):\n        raise ValueError("Both stages must keep the same optimizer effective batch size")\n    if settings.get("refit_full_training") is not True:\n        raise ValueError("ver14 always refits its one selected candidate on all Training")\n    if type(settings.get("seed")) is not int or settings["seed"] < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    if not 0 < settings.get("holdout_size", 0) < 1:\n        raise ValueError("holdout_size must lie strictly between zero and one")\n    for key in ("learning_rate", "head_learning_rate"):\n        if (isinstance(settings.get(key), bool) or not isinstance(settings.get(key), Real)\n                or not math.isfinite(settings[key]) or settings[key] <= 0):\n            raise ValueError(f"{key} must be positive and finite")\n    if type(settings.get("overlap")) is not int or not 0 <= settings["overlap"] < settings["max_length"] - 2:\n        raise ValueError("overlap must fit within a token window")\n    if not isinstance(settings.get("resume_run", ""), str):\n        raise ValueError("resume_run must be an empty string or a prior ver14 run path")\n    return settings\n\n\ndef _candidate(settings, name, stage, weight, attention, contrastive, batch, accumulation):\n    return {**settings, "candidate_id": name, "stage": stage,\n            "ce_views": 1 if stage == 2 else 2,\n            "nausea_positive_weight": weight, "label_attention": attention,\n            "contrastive_weight": contrastive, "batch_size": batch,\n            "gradient_accumulation": accumulation, "refit_full_training": False,\n            "run_validation": False, "resume_run": ""}\n\n\ndef stage2_plan(settings):\n    return [_candidate(settings, "stage2_ce_w" + f"{weight:g}".replace(".", "p"), 2,\n                       weight, False, 0.0, settings["batch_size"], settings["gradient_accumulation"])\n            for weight in settings["nausea_weight_candidates"]]\n\n\ndef stage3_plan(settings, selected_weight):\n    return [_candidate(settings, name, 3, selected_weight, attention, contrastive,\n                       settings["stage3_batch_size"], settings["stage3_gradient_accumulation"])\n            for name, attention, contrastive in (\n                ("stage3_attention_off_ce", False, 0.0),\n                ("stage3_attention_on_ce", True, 0.0),\n                ("stage3_attention_on_supcon", True, 0.1))]\n\n\ndef select_candidate(records):\n    """Select by Training only; a newly trained baseline remains an eligible control."""\n    baseline = next((r for r in records if r["candidate_id"] == BASELINE), None)\n    if baseline is None:\n        raise ValueError("The unweighted Training baseline must be evaluated first")\n    base_macro = baseline["metrics"]["macro_f1"]\n    base_nausea = baseline["metrics"]["per_symptom"][NAUSEA]["f1"]\n    eligible = []\n    for record in records:\n        metrics = record["metrics"]\n        macro, nausea = metrics["macro_f1"], metrics["per_symptom"][NAUSEA]["f1"]\n        if not all(isinstance(x, Real) and math.isfinite(x) and 0 <= x <= 1 for x in (macro, nausea)):\n            raise ValueError("Candidate selection metrics must be finite F1 scores")\n        if macro >= base_macro and nausea >= base_nausea:\n            eligible.append(record)\n    return max(eligible, key=lambda r: (r["metrics"]["macro_f1"], r["metrics"]["per_symptom"][NAUSEA]["f1"]))\n\n\ndef _manifest(value):\n    return {**value, "format": "mission3_categorical_v1", "labels": list(data.TARGET_SYMPTOMS),\n            "version": "ver14", "decision_rule": "categorical predict/argmax"}\n\n\ndef _training_settings(settings):\n    return {k: v for k, v in settings.items() if k not in {\n        "input_zip", "train_dir", "validation_dir", "work_dir", "output_dir", "resume_run", "log_every"}}\n\n\ndef _run_candidate(settings, paths, texts, targets, fit, holdout, root):\n    name = settings["candidate_id"]\n    candidate_root = root / "experiments" / name\n    completed = candidate_root / "completed.json"\n    signature = _digest(_training_settings(settings))\n    if completed.is_file():\n        record = _read(completed)\n        if (record["settings_digest"] != signature or _digest(record["settings"]) != signature\n                or record["settings"] != _training_settings(settings)\n                or record["candidate_id"] != name or record["stage"] != settings["stage"]):\n            raise ValueError("Resume candidate configuration mismatch")\n        if Path(record["directory"]).parts[:2] != ("experiments", name):\n            raise ValueError("Resume candidate directory mismatch")\n        _verify_hashes(root, record["artifact_hashes"])\n        local_manifest = _read(_safe(root, record["directory"] + "/candidate_model.ckpt"))\n        expected_manifest = {**local_manifest,\n            "weights": record["directory"] + "/" + local_manifest["weights"],\n            "assets": record["directory"] + "/" + local_manifest["assets"]}\n        if record["manifest"] != expected_manifest:\n            raise ValueError("Resume candidate manifest mismatch")\n        print(f"Reusing completed Training candidate: {name}", flush=True)\n    else:\n        attempt = candidate_root / _stamp()\n        attempt.mkdir(parents=True, exist_ok=False)\n        local = Path(settings["work_dir"]) / root.name / name / attempt.name\n        local.mkdir(parents=True, exist_ok=False)\n        data.write_json(attempt / "settings.json", settings)\n        print(f"Stage {settings[\'stage\']}: {name}; nausea positive weight={settings[\'nausea_positive_weight\']:g}", flush=True)\n        started = time.monotonic()\n        manifest, metrics = runtime.train_neural(paths, texts, targets, fit, holdout, local, attempt, settings)\n        manifest = _manifest(manifest)\n        data.write_json(attempt / "candidate_model.ckpt", manifest)\n        data.write_json(attempt / "training_metrics.json", metrics)\n        relative = attempt.relative_to(root).as_posix()\n        root_manifest = {**manifest, "weights": f"{relative}/{manifest[\'weights\']}",\n                         "assets": f"{relative}/{manifest[\'assets\']}"}\n        record = {"candidate_id": name, "stage": settings["stage"], "settings_digest": signature,\n                  "settings": _training_settings(settings), "manifest": root_manifest,\n                  "directory": relative, "metrics": metrics, "seconds": time.monotonic() - started}\n        record["artifact_hashes"] = _hashes(root, root_manifest, (\n            f"{relative}/holdout_predictions.npz", f"{relative}/selection_history.json",\n            f"{relative}/candidate_model.ckpt", f"{relative}/training_metrics.json", f"{relative}/settings.json"))\n        data.write_json(completed, record)\n    with np.load(_safe(root, record["directory"] + "/holdout_predictions.npz"), allow_pickle=False) as values:\n        if not np.array_equal(values["indices"], holdout) or not np.array_equal(values["targets"], targets[holdout]):\n            raise ValueError("Cached candidate targets or split do not match Training")\n        actual = data.score_predictions(targets[holdout], values["predictions"])\n    if actual != record["metrics"]:\n        raise ValueError("Cached candidate metrics do not match its saved discrete predictions")\n    return record\n\n\ndef _write_table(root, records):\n    fields = ["candidate", "stage", "nausea_positive_weight", "label_attention", "contrastive_weight",\n              "batch_size", "ce_views", "selected_epochs", "macro_f1", "nausea_f1", "nausea_precision",\n              "nausea_recall", "nausea_fp", "nausea_fn", "seconds"]\n    temporary = root / "candidate_summary.csv.tmp"\n    with temporary.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        for r in records:\n            n = r["metrics"]["per_symptom"][NAUSEA]\n            writer.writerow({"candidate": r["candidate_id"], "stage": r["stage"],\n                             **{k: r["settings"][k] for k in fields[2:6]},\n                             "batch_size": r["settings"]["batch_size"],\n                             "ce_views": r["settings"]["ce_views"],\n                             "selected_epochs": r["manifest"]["selected_epochs"],\n                             "macro_f1": r["metrics"]["macro_f1"], "nausea_f1": n["f1"],\n                             "nausea_precision": n["precision"], "nausea_recall": n["recall"],\n                             "nausea_fp": n["false_positive"], "nausea_fn": n["false_negative"],\n                             "seconds": r["seconds"]})\n    temporary.replace(root / "candidate_summary.csv")\n\n\ndef _save_environment(root):\n    import torch\n    from importlib.metadata import version, PackageNotFoundError\n    packages = {}\n    for name in ("torch", "transformers", "numpy", "scikit-learn", "tokenizers"):\n        try:\n            packages[name] = version(name)\n        except PackageNotFoundError:\n            packages[name] = "not installed"\n    data.write_json(root / ("environment_" + _stamp() + ".json"), {\n        "python": platform.python_version(), "packages": packages, "cuda_available": torch.cuda.is_available(),\n        "cuda_version": torch.version.cuda,\n        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})\n\n\ndef _final_model(settings, selected, texts, targets, root):\n    completed = root / "final_completion.json"\n    checkpoint = root / "mission3_ver14_model.ckpt"\n    if completed.is_file():\n        saved = _read(completed)\n        if saved["candidate_id"] != selected["candidate_id"]:\n            raise ValueError("Final candidate changed during resume")\n        _verify_hashes(root, saved["artifact_hashes"])\n        return checkpoint\n    attempt = root / "final_model" / _stamp()\n    attempt.mkdir(parents=True, exist_ok=False)\n    final_settings = {**settings, **selected["settings"], "refit_full_training": True}\n    data.write_json(attempt / "settings.json", final_settings)\n    manifest, expected_probe = runtime.refit_selected(\n        texts, targets, final_settings, selected["manifest"]["selected_epochs"], attempt)\n    manifest = _manifest(manifest)\n    local_checkpoint = attempt / "mission3_ver14_model.ckpt"\n    data.write_json(local_checkpoint, manifest)\n    actual_probe = runtime.predict_checkpoint(local_checkpoint, texts[:len(expected_probe)])\n    if not np.array_equal(expected_probe, actual_probe):\n        raise ValueError("Reloaded final model predictions differ from its in-memory model")\n    relative = attempt.relative_to(root).as_posix()\n    manifest["weights"] = f"{relative}/{manifest[\'weights\']}"\n    manifest["assets"] = f"{relative}/{manifest[\'assets\']}"\n    manifest["selected_candidate"] = selected["candidate_id"]\n    data.write_json(checkpoint, manifest)\n    data.write_json(root / "final_history.json", _read(attempt / "final_history.json"))\n    data.write_json(completed, {"candidate_id": selected["candidate_id"],\n                               "reloaded_probe_samples": len(expected_probe),\n                               "artifact_hashes": _hashes(root, manifest, (checkpoint.name, "final_history.json"))})\n    return checkpoint\n\n\ndef run(settings, runtime_sources=None):\n    started = time.monotonic()\n    settings = validate_settings(settings)\n    sources = runtime_sources or {p.name: p.read_text(encoding="utf-8") for p in Path(__file__).parent.iterdir()\n                                 if p.is_file() and p.suffix in (".py", ".md", ".txt")}\n    sources = {k: v for k, v in sources.items() if not k.startswith("test_") and k != "build_notebook.py"}\n    # Source and settings fingerprints make resuming completed candidates explicit and reproducible.\n    source_hash = _digest({k: v for k, v in sources.items() if not k.endswith(".md")})\n    settings_hash = _digest(_training_settings(settings))\n    output_base = Path(settings["output_dir"]).expanduser().resolve()\n    resume = settings.get("resume_run", "")\n    root = Path(resume).expanduser().resolve() if resume else output_base / _stamp()\n    if root == output_base or not root.is_relative_to(output_base):\n        raise ValueError("resume_run must be a run directory inside output_dir")\n    if resume and not (root / "run_identity.json").is_file():\n        raise ValueError("resume_run is not an initialized ver14 run")\n    if not resume:\n        root.mkdir(parents=True, exist_ok=False)\n    print("Result run:", root, flush=True)\n    print("Resume a disconnected session by setting resume_run to this exact directory.", flush=True)\n    can_write_status = not bool(resume)\n    try:\n        train_dir, validation_dir = runtime.prepare_data(settings["input_zip"], settings["work_dir"],\n                                                        settings.get("train_dir", ""), settings.get("validation_dir", ""))\n        paths, texts, targets = data.load_dataset(train_dir, include_targets=True)\n        fit, holdout = data.make_split(texts, targets, settings["holdout_size"], settings["seed"])\n        identity = {"version": "ver14", "training_digest": runtime.dataset_digest(texts, targets),\n                    "source_digest": source_hash, "settings_digest": settings_hash,\n                    "training_samples": len(texts), "fit_samples": len(fit), "holdout_samples": len(holdout),\n                    "model_selection": "Training internal holdout only", "decision_rule": "categorical predict/argmax",\n                    "selection_policy": POLICY}\n        if resume:\n            if _read(root / "run_identity.json") != identity:\n                raise ValueError("Resume source, data, settings, or selection protocol changed")\n            with np.load(root / "training_split.npz", allow_pickle=False) as split:\n                if not np.array_equal(split["fit"], fit) or not np.array_equal(split["holdout"], holdout):\n                    raise ValueError("Resume Training split changed")\n            can_write_status = True\n        else:\n            np.savez_compressed(root / "training_split.npz", fit=fit, holdout=holdout)\n            data.write_json(root / "run_identity.json", identity)\n            data.write_json(root / "settings.json", settings)\n            submission = root / "submission"\n            submission.mkdir()\n            for name, source in sources.items():\n                if Path(name).name != name:\n                    raise ValueError("Runtime source names must be flat files")\n                (submission / name).write_text(source, encoding="utf-8")\n        if (root / "run_complete.json").is_file():\n            _verify_hashes(root, _read(root / "run_complete.json")["artifact_hashes"])\n            print("This run is already complete; verified its final artifacts.", flush=True)\n            return root\n        _save_environment(root)\n        audit_path = root / "audit_stage.json"\n        if not audit_path.exists():\n            print("Stage 1: Training evidence audit; original targets are retained.", flush=True)\n            data.write_json(audit_path, write_audit(paths, texts, targets, fit, holdout, root / "audit"))\n            data.write_json(root / "training_diagnostics.json", data.training_diagnostics(texts, targets))\n        records = []\n        plan = {"stage1": "Training evidence audit without relabeling or sample removal",\n                "stage2": stage2_plan(settings), "stage3": "Matched ablations using the selected stage2 weight",\n                "selection_policy": POLICY, "official_validation": "Only after final selection, refit and CSV save"}\n        data.write_json(root / "experiment_plan.json", plan)\n        for candidate in plan["stage2"]:\n            records.append(_run_candidate(candidate, paths, texts, targets, fit, holdout, root))\n            _write_table(root, records)\n        stage2_winner = select_candidate(records)\n        data.write_json(root / "stage2_selection.json", {"candidate_id": stage2_winner["candidate_id"],\n                        "selection_policy": POLICY, "metrics": stage2_winner["metrics"]})\n        plan["stage3"] = stage3_plan(settings, stage2_winner["settings"]["nausea_positive_weight"])\n        data.write_json(root / "experiment_plan.json", plan)\n        for candidate in plan["stage3"]:\n            records.append(_run_candidate(candidate, paths, texts, targets, fit, holdout, root))\n            _write_table(root, records)\n        selected = select_candidate(records)\n        data.write_json(root / "final_selection.json", {"candidate_id": selected["candidate_id"],\n                        "selection_policy": POLICY, "metrics": selected["metrics"],\n                        "selected_epochs": selected["manifest"]["selected_epochs"],\n                        "baseline_selected": selected["candidate_id"] == BASELINE,\n                        "candidate_settings": selected["settings"]})\n        data.write_json(root / "training_metrics.json", selected["metrics"])\n        saved = _safe(root, selected["directory"])\n        shutil.copy2(saved / "holdout_predictions.npz", root / "holdout_predictions.npz")\n        with np.load(root / "holdout_predictions.npz", allow_pickle=False) as values:\n            data.export_errors([paths[i] for i in holdout], [texts[i] for i in holdout], targets[holdout],\n                               values["predictions"], root / "training_errors")\n        print("Selected from Training:", selected["candidate_id"], flush=True)\n        checkpoint = _final_model(settings, selected, texts, targets, root)\n        completion = _read(root / "final_completion.json")["artifact_hashes"]\n        if settings.get("run_validation", True):\n            validation_paths, predicted = runtime.inference(checkpoint, validation_dir, root / "mission3.csv")\n            # This is the first use of official targets. They never affect model/epoch/weight selection.\n            scored_paths, _, truth = data.load_dataset(validation_dir, include_targets=True)\n            if [p.name for p in scored_paths] != [p.name for p in validation_paths]:\n                raise ValueError("Official evaluation row order mismatch")\n            metrics = data.score_predictions(truth, predicted)\n            data.write_json(root / "validation_metrics.json", metrics)\n            for name in ("mission3.csv", "validation_metrics.json"):\n                completion[name] = _hash(root / name)\n            print(f"Final Validation Macro F1={metrics[\'macro_f1\']:.6f}; nausea F1={metrics[\'per_symptom\'][NAUSEA][\'f1\']:.6f}", flush=True)\n        data.write_json(root / "runtime.json", {"status": "complete", "seconds": time.monotonic() - started,\n                        "seconds_scope": "current invocation; candidate times are in candidate_summary.csv",\n                        "candidate_count": len(records), "selected_candidate": selected["candidate_id"]})\n        data.write_json(root / "run_complete.json", {"artifact_hashes": completion})\n    except BaseException as error:\n        if can_write_status:\n            data.write_json(root / "runtime.json", {"status": "interrupted" if isinstance(error, (KeyboardInterrupt, SystemExit)) else "failed",\n                            "seconds": time.monotonic() - started, "error_type": type(error).__name__})\n        raise\n    return root\n', 'inference.py': '"""Official one-command inference entry point for a portable result folder."""\nimport importlib.util\nfrom pathlib import Path\nimport sys\n\nif __package__:\n    from .runtime import main\nelse:\n    root = Path(__file__).resolve().parent\n    name = "mission3_categorical_submission"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    from importlib import import_module\n    main = import_module(name + ".runtime").main\n\nif __name__ == "__main__":\n    main()\n', 'run_notebook.py': '"""Launch the ver14 experiment suite in a fresh interpreter after installation."""\nfrom __future__ import annotations\n\nimport argparse\nimport importlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport sys\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--settings", required=True)\n    parser.add_argument("--result_file", required=True)\n    args = parser.parse_args(argv)\n    root = Path(__file__).resolve().parent\n    name = "mission3_ver14_execution"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    suite = importlib.import_module(name + ".runtime_suite")\n    settings = json.loads(Path(args.settings).read_text(encoding="utf-8-sig"))\n    sources = {path.name: path.read_text(encoding="utf-8") for path in sorted(root.iterdir())\n               if path.is_file() and (path.suffix in (".py", ".md") or path.name == "requirements.txt")}\n    result = suite.run(settings, runtime_sources=sources)\n    Path(args.result_file).write_text(json.dumps(str(result)), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\nsentencepiece>=0.2,<0.3\nprotobuf>=4.25,<7\n', 'README.md': '# Mission 3 ver14 — 오심 분석과 단계별 학습 비교\n\n`../mission3_ver14.ipynb` 한 개를 Colab에서 열고 기존 `mission3_ver5_inputs.zip`으로 실행합니다.\n실행 코드와 requirements가 노트북에 포함되어 다른 Python 파일 업로드가 필요 없습니다.\n이 버전은 기존 ver10–ver13 코드와 결과를 수정하지 않습니다.\n\n## 실행\n\n1. ZIP 바로가기가 있는 공용 계정으로 Colab에 로그인하고 GPU 런타임을 선택합니다.\n2. `런타임 → 모두 실행`을 선택합니다. ZIP 기본 경로는 `/content/drive/MyDrive/mission3_ver5_inputs.zip`입니다.\n3. 결과는 `MyDrive/DCC_AIdeas/results/mission3/ver14/실행시각_고유번호/`에 저장됩니다.\n4. Training 후보 비교와 최종 선택 근거를 먼저 확인하고, 마지막 Validation 결과를 기록합니다.\n\n기본 실행은 후보 6개 학습과 최종 구성 1개 전체 Training 재학습을 수행합니다.\n비슷한 GPU에서 약 6–10시간을 예상하지만 GPU 성능·메모리·조기 종료에 따라 달라집니다.\n고정 시간 제한은 없습니다. **GPU 실행 시간과 F1 개선은 실제 실행 전에는 확인되지 않았습니다.**\n\n## 1. 원본과 정답 점검\n\n기존 오심 검토 표 80행, 중복을 제외한 76개 통화를 원본과 대조했습니다.\n개인정보를 제외한 관찰 사례는 다음과 같습니다.\n\n| 증상 내용의 요약 | 제공된 정답과 모델 결과 | 검토 의미 |\n| --- | --- | --- |\n| 메스꺼움과 구토를 함께 말함 | 정답은 구토·어지러움, 모델 오심은 FP | 명시 표현과 정답 조합이 어긋나는 사례 |\n| 토할 것 같은 느낌과 어지러움을 말함 | 정답은 어지러움, 모델 오심은 FP | 구토·오심·어지러움의 경계 |\n| 토할 것 같은 느낌을 말함 | 정답에 오심이 포함되지만 모델은 구토만 예측하여 FN | 유사 표현에서 발생한 증상 혼동 |\n\n관찰만으로 라벨 오류를 확정하지 않습니다. 원본 정답을 그대로 유지하며,\n질문·부정·간접 증상 표현과 정답 조합을 사람이 검토합니다.\n새 실행은 Training 데이터에서 fit/holdout을 구분해 분포·오심/구토 조합·검토 후보를 저장합니다.\n1단계 검토 목록은 원본 행 위치·정답·후보 표시를 포함하고 화면에는 집계만 표시합니다.\n학습 후 오류 CSV에는 실제 전사가 저장됩니다.\n\n기존 ver13의 오심 FN 575개 중 567개(98.6%)는 단일 512-token window였습니다.\nver11에서 맞혔다가 ver13이 놓친 16개도 두 tokenizer 모두 단일 window였고,\nKcELECTRA 기준 최대 417 tokens였습니다. 정규식으로 찾은 오심 표현 후보는 모두 첫 window에 있었습니다.\n이 결과는 긴 대화 확장보다 정답의 경계와 학습 손실을 먼저 점검할 근거입니다.\n정규식에 없는 우회 표현이나 실제 정답의 정확성은 별도로 검토해야 합니다.\n\n## 2. 오심 양성 학습 가중치 비교\n\nKLUE-RoBERTa-base의 공개 revision `02f94ba5e3fcb7e2a58a390b8639b0fac974a8da`를 고정합니다.\nattention 없이 교차엔트로피를 학습하고 오심 양성 class의 가중치 1·2·3을 비교합니다.\n가중치 1이 이번 실행의 새 기준 모델입니다. 가중치는 학습 중 오분류 비용에 적용하며,\n최종 예측은 두 class logits의 argmax를 그대로 사용합니다.\n오심 재현율이 늘어도 오탐이나 다른 증상이 악화될 수 있으므로 오심 P/R/F1과 전체 Macro F1을 함께 비교합니다.\n\n## 3. Attention과 대조학습을 분리해 비교\n\n2단계에서 선택한 가중치를 고정하고 다음 세 후보를 새로 학습합니다.\n\n| 후보 | Attention | 교차엔트로피 | 대조학습 |\n| --- | --- | --- | --- |\n| attention_off_ce | 끔 | 사용 | 없음 |\n| attention_on_ce | 켬 | 사용 | 없음 |\n| attention_on_supcon | 켬 | 사용 | 0.1 |\n\n3단계는 세 후보 모두 실제 batch 8, gradient accumulation 2입니다.\n세 후보 모두 동일 통화를 dropout으로 두 번 읽고 두 교차엔트로피의 평균을 사용합니다.\n`attention_on_supcon`에만 대조학습을 더해 두 view 사용과 대조학습 손실의 효과를 분리합니다.\n2단계는 batch 4, accumulation 4이며 유효 batch는 두 단계 모두 16입니다.\n대조학습은 실제 batch 안의 표본만 비교하며 accumulation이 비교집합을 늘리지는 않습니다.\nencoder·Training 분할·seed·학습률·epoch 선택 규칙을 맞춰 비교합니다.\n\n최종 선택은 이번 실행의 새 기준 모델 대비 **Macro F1과 오심 F1을 모두 유지하거나 개선한 후보**에 한정하고,\n그 안에서 Macro F1을 우선합니다. 개선 후보가 조건을 통과하지 못하면 새 기준 모델을 유지합니다.\n선택된 구성 하나만 공개 초기 가중치에서 전체 Training으로 다시 학습하고,\n공식 Validation은 마지막 예측 CSV를 저장한 뒤 한 번 평가합니다.\n\n## 실행 조건과 재개\n\n- 입력은 양측 발화의 `text`만 사용합니다. symptom은 정답으로만 사용합니다.\n- 중복 전사 그룹이 내부 fit/holdout에 섞이지 않게 분할합니다. Validation은 학습·후보 선택에 쓰지 않습니다.\n- 추가 corpus·상용 AI API·사후 threshold 검색·calibration·기존 모델 fallback을 사용하지 않습니다.\n- 학습 가중치는 미리 정한 1·2·3만 비교합니다. 검토 표현으로 정답을 수정하거나 샘플을 제거하지 않습니다.\n- 모든 window를 유지하며 증상별 두 class의 argmax로 0개 또는 여러 증상을 예측합니다.\n- dependency 버전 범위는 기존 ver10–ver13 requirements와 같습니다.\n\n중단 후에는 노트북 설정의 `resume_run`에 이미 생성된 실행 폴더의 전체 Drive 경로를 지정합니다.\n코드·Training 데이터·설정 fingerprint를 확인한 뒤 완료된 후보 결과를 재사용합니다.\n미완료 후보의 epoch 중간 상태는 이어서 학습하지 않습니다. 해당 후보는 새 시도에서 처음부터 학습합니다.\n설정이 다른 실험은 `resume_run`을 비우고 새로 실행합니다.\nGPU 메모리가 부족하면 해당 단계의 batch를 줄이고 accumulation을 맞춰 조절하세요.\n실제 batch 변경은 대조학습 비교 표본에도 영향을 주므로 서로 다른 조건의 실험을 그대로 재개하지 않습니다.\n\n## 결과와 제출\n\n| 파일 | 내용 |\n| --- | --- |\n| `experiment_plan.json` | Training 기반 단계별 실험 조건 |\n| `candidate_summary.csv` | 6개 후보의 오심 P/R/F1·FP/FN과 전체 Macro F1 |\n| `final_selection.json` | 최종 선택과 기준 모델 대비 조건 충족 여부 |\n| `training_metrics.json` | Training holdout에서 선택한 구성의 지표 |\n| `validation_metrics.json` | 선택·최종 재학습 완료 후 공식 Validation 평가 |\n| `runtime.json` | 실행 시간과 완료 상태 |\n| `mission3.csv` | 최종 argmax 예측, `label file name,symptom` 헤더 |\n| `mission3_ver14_model.ckpt`, 가중치·`assets/`·`submission/` | 이동 가능한 최종 추론 묶음 |\n\n`.ckpt`는 manifest입니다. 같은 실행 폴더의 실제 가중치와 assets를 함께 보관해야 합니다.\n최종 가중치와 assets는 `실행폴더/final_model/시도번호/` 아래에 저장되며,\n실행 폴더 루트의 `mission3_ver14_model.ckpt`는 해당 파일들의 상대 경로를 참조합니다.\n`runtime.json`의 seconds는 재개를 포함한 이번 실행 호출의 시간이며 후보별 시간은 `candidate_summary.csv`에 기록됩니다.\n실행 로그가 저장된 학습 노트북과 실제 결과 PPT도 준비합니다.\n\n```text\npython submission/inference.py --audio_dir /path/audio --label_dir /path/json --ckpt_path mission3_ver14_model.ckpt --output mission3.csv\n```\n\n노트북을 다시 생성하려면 다음을 실행합니다. ver14 파일만 생성합니다.\n\n```text\npython src/mission3/mission3_ver14/build_notebook.py\n```\n'}
runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver14_"))
for name, source in RUNTIME_SOURCES.items():
    (runtime_dir / name).write_text(source, encoding="utf-8")
print("실행 코드 준비 완료")


In [ ]:
# @title 기존 프로젝트와 같은 버전의 패키지 준비
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "-r", str(runtime_dir / "requirements.txt")])


In [ ]:
# @title 1. 원본 점검 → 2. 가중치 비교 → 3. Attention·대조학습 비교 → 최종 학습·평가
import json
settings_file = runtime_dir / "notebook_settings.json"
result_file = runtime_dir / "notebook_result.json"
settings_file.write_text(json.dumps(SETTINGS, ensure_ascii=False, allow_nan=False), encoding="utf-8")
# A fresh interpreter uses the installed versions and streams progress to this cell.
subprocess.check_call([sys.executable, "-u", str(runtime_dir / "run_notebook.py"),
                       "--settings", str(settings_file), "--result_file", str(result_file)])
result_dir = Path(json.loads(result_file.read_text(encoding="utf-8")))


In [ ]:
# @title 비교 결과와 최종 점수
import csv
import json
summary = result_dir / "candidate_summary.csv"
if summary.is_file():
    print("Training 내부 비교:")
    with summary.open(encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle):
            print(row["candidate"], "Macro F1 =", round(float(row["macro_f1"]), 6),
                  "오심 F1 =", round(float(row["nausea_f1"]), 6),
                  "P =", round(float(row["nausea_precision"]), 6),
                  "R =", round(float(row["nausea_recall"]), 6),
                  "FP/FN =", row["nausea_fp"], "/", row["nausea_fn"])
for filename in ["training_metrics.json", "validation_metrics.json"]:
    path = result_dir / filename
    if path.is_file():
        report = json.loads(path.read_text(encoding="utf-8-sig"))
        nausea = report["per_symptom"]["오심"]
        print(filename, "Macro F1 =", round(report["macro_f1"], 6),
              "오심 F1 =", round(nausea["f1"], 6),
              "P =", round(nausea["precision"], 6), "R =", round(nausea["recall"], 6))
elapsed = result_dir / "runtime.json"
if elapsed.is_file():
    report = json.loads(elapsed.read_text(encoding="utf-8-sig"))
    if "seconds" in report:
        print("이번 실행 소요 시간:", round(report["seconds"] / 3600, 2), "시간")
print("결과 폴더:", result_dir)
print("후보 비교:", result_dir / "candidate_summary.csv")
print("최종 선택 근거:", result_dir / "final_selection.json")
print("최종 예측:", result_dir / "mission3.csv")


## 중단 후 재개와 제출

기본 실행은 `ver14/실행시각_고유번호/`에 저장됩니다. 기존 ver10–ver13 결과는 유지합니다.
중단되면 설정의 `resume_run`에 그 실행 폴더의 전체 Drive 경로를 넣고 다시 실행하세요.
코드·Training 데이터·설정의 fingerprint가 같을 때 완료된 후보 결과를 재사용합니다.
다른 설정의 실험은 `resume_run`을 비우고 새로 실행합니다.

`experiment_plan.json`, `candidate_summary.csv`, `final_selection.json`,
`training_metrics.json`, `validation_metrics.json`, `runtime.json`을 함께 확인하세요.
`.ckpt`는 manifest이므로 **같은 실행 폴더의 실제 가중치·assets·submission 코드 전체**를 보관합니다.
최종 가중치와 assets는 `final_model/시도번호/`에 있고 루트 `.ckpt`가 그 상대 경로를 참조합니다.
중단된 후보의 epoch 중간 상태는 재개하지 않으며 그 후보는 새 시도에서 처음부터 학습합니다.
실행 로그가 포함된 이 노트북도 저장해야 합니다. 제출 추론 명령:

```bash
python submission/inference.py --audio_dir /path/audio --label_dir /path/json --ckpt_path mission3_ver14_model.ckpt --output mission3.csv
```

추론은 저장된 로컬 모델로 실행하고 CSV 헤더는 `label file name,symptom`입니다.
검토 CSV는 실제 통화 전사를 포함하므로 오류 분석용으로 관리하세요.
실제 결과 PPT는 GPU 학습과 마지막 평가를 마친 뒤 작성합니다.
